
# 13 - Final Broad + Fine XAI Matrix

Reload-first XAI notebook for the thesis. This notebook is the current notebook-side source of truth for the XAI layer: it keeps the full `M0-M4` mode grammar visible, separates `historical_weather_memory` from `future_weather`, and interprets `FW0`, `FW2`, and `FW1` respectively as the clean-identification, realistic forecast-like, and oracle upper-bound weather settings.

## What Changed In The XAI Layer

The notebook now reads grouped explainability through two linked vocabularies rather than one. The broad thesis rollup keeps the original headline language for the chapter-level story, while the fine grouped taxonomy adds a more diagnostic decomposition that is easier to defend when feature blocks are codependent.

## Broad Thesis Rollup vs Fine Diagnostic Taxonomy

The broad rollup is still the main thesis-facing reading layer because it keeps the argument compact: demand history, current weather, historical weather memory, future weather, system dynamics, calendar, and static building context. The fine taxonomy keeps the same logic but splits the broad groups into cleaner internal pieces such as future-weather summaries versus future-weather paths, space-heating dynamics versus ventilation dynamics, and static profile features versus EHR-derived morphology. Grouped interpretation remains the default because the retained feature space contains lagged, rolled, and structurally dependent variables that should not be over-read as isolated single-feature truths.


In [1]:
from __future__ import annotations

import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path


def ensure_runtime_packages(dep_to_package: dict[str, str]) -> None:
    missing_packages = [pkg for mod, pkg in dep_to_package.items() if importlib.util.find_spec(mod) is None]
    if missing_packages:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
        importlib.invalidate_caches()


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    project_root_override = os.environ.get('THESIS_PROJECT_ROOT', '').strip()

    candidates: list[Path] = []
    if project_root_override:
        candidates.append(Path(project_root_override).expanduser())
    candidates.extend([cwd, cwd / 'thesis-project', cwd.parent / 'thesis-project'])

    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'data' / 'features' / 'feature_metadata.csv').exists():
            return candidate

    raise FileNotFoundError(
        'Could not locate the thesis-project root. Set THESIS_PROJECT_ROOT or start the runtime '
        'in the project directory so that `data/features/feature_metadata.csv` is available.'
    )


In [2]:
PROJECT_ROOT = resolve_project_root()
os.environ['THESIS_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('ABSL_MIN_LOG_LEVEL', '3')
os.environ.setdefault('MPLCONFIGDIR', str((PROJECT_ROOT / '.mplconfig').resolve()))

ensure_runtime_packages(
    {
        'numpy': 'numpy',
        'pandas': 'pandas',
        'matplotlib': 'matplotlib',
        'seaborn': 'seaborn',
        'sklearn': 'scikit-learn',
        'xgboost': 'xgboost',
        'shap': 'shap',
    }
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import utils.model_family_comparison as mfc
import utils.xai_notebook_support as xns

try:
    import absl.logging as absl_logging
    absl_logging.set_verbosity(absl_logging.ERROR)
except Exception:
    pass

try:
    mfc.tf.get_logger().setLevel('ERROR')
except Exception:
    pass


def pick_existing_results_dir(candidates: list[Path], artifact_name: str) -> Path:
    for candidate in candidates:
        candidate = Path(candidate).resolve()
        if (candidate / artifact_name).exists():
            return candidate
    raise FileNotFoundError(
        f'Could not find any saved results directory containing `{artifact_name}`. Checked: '
        + ', '.join(str(Path(candidate).resolve()) for candidate in candidates)
    )


REPORT_ROOT = PROJECT_ROOT / 'results' / 'report_ready_20260405'
MODEL_FAMILY_REPORT_BASE_DIR = REPORT_ROOT / 'model_family_base'
MODEL_FAMILY_REPORT_FINAL_DIR = REPORT_ROOT / 'model_family'
MODEL_FAMILY_REPORT_FULL_DIR = REPORT_ROOT / 'model_family_m3_supplement'
RAW_COMPARISON_RESULTS_DIR = PROJECT_ROOT / 'results' / 'model_family_comparison_29032026'
COMPARISON_RESULTS_DIR = pick_existing_results_dir(
    [MODEL_FAMILY_REPORT_FULL_DIR, MODEL_FAMILY_REPORT_FINAL_DIR, MODEL_FAMILY_REPORT_BASE_DIR, RAW_COMPARISON_RESULTS_DIR],
    'comparison_metrics.csv',
)
comparison_metrics_df = pd.read_csv(COMPARISON_RESULTS_DIR / 'comparison_metrics.csv')

RAW_XAI_RESULTS_DIR = PROJECT_ROOT / 'results' / 'xai_final_31032026'
XAI_REPORT_BASE_DIR = REPORT_ROOT / 'xai_base'
XAI_REPORT_FINAL_DIR = REPORT_ROOT / 'xai'
FULL_MATRIX_RESULTS_DIR = PROJECT_ROOT / 'results' / 'xai_matrix_work_20260405'
FULL_MATRIX_REPORT_READY_DIR = FULL_MATRIX_RESULTS_DIR / '_report_ready'
LOAD_RESULTS_DIR = pick_existing_results_dir(
    [FULL_MATRIX_REPORT_READY_DIR, XAI_REPORT_FINAL_DIR, XAI_REPORT_BASE_DIR, RAW_XAI_RESULTS_DIR],
    'xai_seed_metrics.csv',
)
# Optional extra result directories to merge into the notebook view.
MERGE_SOURCE_RESULTS_DIRS: tuple[Path, ...] = ()
AUTO_EXPORT_REPORT_READY = True

XAI_WORKFLOW_PROFILE = 'resume_fine_backfill_all_modes'
VALID_XAI_WORKFLOW_PROFILES = (
    'full_matrix_review',
    'resume_m3_missing_only',
    'resume_fine_backfill_all_modes',
)
if XAI_WORKFLOW_PROFILE not in VALID_XAI_WORKFLOW_PROFILES:
    raise ValueError(f'Unknown XAI_WORKFLOW_PROFILE: {XAI_WORKFLOW_PROFILE}')

TARGET_KIND = 'cum'
RUN_XGBOOST_FAMILY = True
RUN_LSTM_FAMILY = False
MODEL_FAMILIES = tuple(
    family
    for family, enabled in (
        ('xgboost', RUN_XGBOOST_FAMILY),
        ('lstm', RUN_LSTM_FAMILY),
    )
    if enabled
)
assert len(MODEL_FAMILIES) > 0, 'At least one model family must be enabled.'

THESIS_MODE_GRAMMAR = tuple(xns.THESIS_MODE_GRAMMAR)
THESIS_WEATHER_MODE_ORDER = tuple(xns.THESIS_WEATHER_MODE_ORDER)
REQUESTED_MODES = THESIS_MODE_GRAMMAR
REQUESTED_WEATHER_MODES = THESIS_WEATHER_MODE_ORDER
REGIMES = ('per_building', 'pooled_same_buildings')

available_mode_set = set(comparison_metrics_df['mode'].dropna().astype(str))
available_weather_set = set(comparison_metrics_df['weather_mode'].dropna().astype(str))
AVAILABLE_COMPARISON_MODES = tuple(mode for mode in REQUESTED_MODES if mode in available_mode_set)
AVAILABLE_COMPARISON_WEATHER_MODES = tuple(mode for mode in REQUESTED_WEATHER_MODES if mode in available_weather_set)
MODES = REQUESTED_MODES
WEATHER_MODES = REQUESTED_WEATHER_MODES

BUILDINGS = tuple(sorted(comparison_metrics_df['building'].dropna().astype(str).unique().tolist()))
HORIZON_GRID = xns.normalize_horizons_for_target_kind(
    tuple(sorted(comparison_metrics_df['horizon_h'].dropna().astype(int).unique().tolist())),
    TARGET_KIND,
)
SEED_LIST = (42, 52, 62)
DETAIL_SEED = SEED_LIST[0]

XAI_PROFILE_SETTINGS = {
    'full_matrix_review': {
        'run_full_matrix': False,
        'load_results_dir': LOAD_RESULTS_DIR,
        'report_export_dir': FULL_MATRIX_REPORT_READY_DIR,
        'full_matrix_results_dir': FULL_MATRIX_RESULTS_DIR,
        'run_modes': REQUESTED_MODES,
        'run_weather_modes': REQUESTED_WEATHER_MODES,
        'run_regimes': REGIMES,
        'run_model_families': MODEL_FAMILIES,
        'run_seed_list': SEED_LIST,
        'manifest_strategy': 'thesis_full_matrix',
    },
    'resume_m3_missing_only': {
        'run_full_matrix': True,
        'load_results_dir': LOAD_RESULTS_DIR,
        'report_export_dir': FULL_MATRIX_REPORT_READY_DIR,
        'full_matrix_results_dir': FULL_MATRIX_RESULTS_DIR,
        'run_modes': ('M3',),
        'run_weather_modes': REQUESTED_WEATHER_MODES,
        'run_regimes': REGIMES,
        'run_model_families': MODEL_FAMILIES,
        'run_seed_list': SEED_LIST,
        'manifest_strategy': 'missing_only',
        'require_fine_groups_for_resume': False,
    },
    'resume_fine_backfill_all_modes': {
        'run_full_matrix': True,
        'load_results_dir': FULL_MATRIX_RESULTS_DIR,
        'report_export_dir': FULL_MATRIX_REPORT_READY_DIR,
        'full_matrix_results_dir': FULL_MATRIX_RESULTS_DIR,
        'run_modes': REQUESTED_MODES,
        'run_weather_modes': REQUESTED_WEATHER_MODES,
        'run_regimes': REGIMES,
        'run_model_families': MODEL_FAMILIES,
        'run_seed_list': SEED_LIST,
        'manifest_strategy': 'missing_only',
        'require_fine_groups_for_resume': True,
    },
}

PROFILE = XAI_PROFILE_SETTINGS[XAI_WORKFLOW_PROFILE]
RUN_FULL_MATRIX_IN_NOTEBOOK = bool(PROFILE['run_full_matrix'])
NOTEBOOK_LOAD_RESULTS_DIR = Path(PROFILE['load_results_dir']).resolve()
NOTEBOOK_REPORT_EXPORT_DIR = Path(PROFILE['report_export_dir']).resolve()
RUN_BUILDINGS = BUILDINGS
RUN_HORIZONS = HORIZON_GRID
RUN_REGIMES = tuple(PROFILE['run_regimes'])
RUN_WEATHER_MODES = tuple(PROFILE['run_weather_modes'])
RUN_MODES = tuple(PROFILE['run_modes'])
RUN_MODEL_FAMILIES = tuple(PROFILE['run_model_families'])
RUN_SEED_LIST = tuple(PROFILE['run_seed_list'])
MANIFEST_STRATEGY = str(PROFILE['manifest_strategy'])
REQUIRE_FINE_GROUPS_FOR_RESUME = bool(PROFILE.get('require_fine_groups_for_resume', False))
RESUME_EXISTING = True
SAVE_AFTER_EACH_SLOT = False
SAVE_EVERY_N_SLOTS = 12
CONTINUE_ON_ERROR = True

ANCHOR_BUILDINGS = ('U05', 'U06')
FOCUS_WEATHER_MODE = 'FW0' if 'FW0' in REQUESTED_WEATHER_MODES else REQUESTED_WEATHER_MODES[0]
INERTIA_GROUPS = ['historical_weather_memory', 'system_dynamic_proxies', 'static_building_context']
WEATHER_STORY_GROUPS = ['historical_weather_memory', 'future_weather', 'system_dynamic_proxies', 'current_weather']
SCENARIO_CASE_TYPES = (
    'cold_peak',
    'morning_recovery',
    'coasting_decline',
    'abrupt_weather_shift',
    'high_error',
    'stability_anchor',
    'stability_match',
)

RUN_MATRIX_SMOKE = False
REQUIRE_SMOKE_PASS = True
DETAIL_HORIZONS = tuple(h for h in (8, 24, 36) if h in HORIZON_GRID)
LOCAL_HORIZONS = tuple(h for h in (8, 24) if h in HORIZON_GRID)
SMOKE_BUILDINGS = tuple(building for building in ANCHOR_BUILDINGS if building in BUILDINGS)
SMOKE_MODES = tuple(mode for mode in ('M0', 'M2', 'M4') if mode in REQUESTED_MODES)
SMOKE_WEATHER_MODES = tuple(mode for mode in ('FW0', 'FW2') if mode in REQUESTED_WEATHER_MODES)
SMOKE_HORIZONS = tuple(h for h in (8, 24) if h in HORIZON_GRID)
PFI_REPEATS_BY_FAMILY = {'xgboost': 10, 'lstm': 3}
SMOKE_PFI_REPEATS_BY_FAMILY = {'xgboost': 2, 'lstm': 1}
SHAP_BACKGROUND_SIZE = 96
SHAP_EXPLAIN_SIZE = 192
LSTM_PFI_MAX_ROWS = 1024

assert set(ANCHOR_BUILDINGS).issubset(set(BUILDINGS)), 'Anchor buildings are not present in the saved comparison outputs.'
assert len(HORIZON_GRID) > 0, 'No cumulative horizons were found in the saved comparison outputs.'
assert len(DETAIL_HORIZONS) > 0, 'No detail horizons remain after cumulative filtering.'
assert len(LOCAL_HORIZONS) > 0, 'No local-case horizons remain after cumulative filtering.'
assert 1 not in HORIZON_GRID, 'Cumulative horizon grid should exclude 1h.'

artifact_paths = xns.build_xai_artifact_paths(NOTEBOOK_REPORT_EXPORT_DIR)
smoke_artifact_paths = xns.build_xai_artifact_paths(FULL_MATRIX_RESULTS_DIR / '_smoke')

config = mfc.ExperimentConfig(
    buildings=BUILDINGS,
    horizons=HORIZON_GRID,
    regimes=REGIMES,
    weather_modes=WEATHER_MODES,
    modes=MODES,
    results_dir=NOTEBOOK_LOAD_RESULTS_DIR,
    xgb_preset_id='P01_md3_lr003_mc5',
    random_seed=DETAIL_SEED,
    model_families=MODEL_FAMILIES,
)

runtime_df = pd.DataFrame(
    [
        {'field': 'project_root', 'value': str(PROJECT_ROOT)},
        {'field': 'xai_workflow_profile', 'value': XAI_WORKFLOW_PROFILE},
        {'field': 'comparison_results_dir', 'value': str(COMPARISON_RESULTS_DIR)},
        {'field': 'load_results_dir', 'value': str(LOAD_RESULTS_DIR)},
        {'field': 'notebook_load_results_dir', 'value': str(NOTEBOOK_LOAD_RESULTS_DIR)},
        {'field': 'notebook_report_export_dir', 'value': str(NOTEBOOK_REPORT_EXPORT_DIR)},
        {'field': 'full_matrix_results_dir', 'value': str(FULL_MATRIX_RESULTS_DIR)},
        {'field': 'target_kind', 'value': TARGET_KIND},
        {'field': 'model_families', 'value': ', '.join(MODEL_FAMILIES)},
        {'field': 'requested_modes', 'value': ', '.join(REQUESTED_MODES)},
        {'field': 'available_comparison_modes', 'value': ', '.join(AVAILABLE_COMPARISON_MODES) if AVAILABLE_COMPARISON_MODES else '(none)'},
        {'field': 'run_modes', 'value': ', '.join(RUN_MODES)},
        {'field': 'manifest_strategy', 'value': MANIFEST_STRATEGY},
        {'field': 'run_weather_modes', 'value': ', '.join(RUN_WEATHER_MODES)},
        {'field': 'weather_modes_present_now', 'value': ', '.join(WEATHER_MODES)},
        {'field': 'seed_list', 'value': ', '.join(str(seed) for seed in SEED_LIST)},
        {'field': 'horizon_grid', 'value': ', '.join(str(h) for h in HORIZON_GRID)},
        {'field': 'detail_horizons', 'value': ', '.join(str(h) for h in DETAIL_HORIZONS)},
        {'field': 'local_horizons', 'value': ', '.join(str(h) for h in LOCAL_HORIZONS)},
        {'field': 'run_matrix_smoke', 'value': str(RUN_MATRIX_SMOKE)},
        {'field': 'run_full_matrix_in_notebook', 'value': str(RUN_FULL_MATRIX_IN_NOTEBOOK)},
        {'field': 'require_smoke_pass', 'value': str(REQUIRE_SMOKE_PASS)},
        {'field': 'resume_existing', 'value': str(RESUME_EXISTING)},
        {'field': 'save_after_each_slot', 'value': str(SAVE_AFTER_EACH_SLOT)},
        {'field': 'save_every_n_slots', 'value': str(SAVE_EVERY_N_SLOTS)},
        {'field': 'continue_on_error', 'value': str(CONTINUE_ON_ERROR)},
    ]
)
artifact_df = pd.DataFrame([{'artifact': key, 'path': str(path)} for key, path in artifact_paths.items()])

base_frames = None
smoke_artifacts = {}
matrix_artifacts = {}
saved_outputs = {}
manifest_df = pd.DataFrame()
full_manifest_df = pd.DataFrame()
desired_full_manifest_df = pd.DataFrame()
current_manifest_df = pd.DataFrame()
xai_base_manifest_df = pd.DataFrame()
xai_inventory_df = pd.DataFrame()
xai_inventory_summary_df = pd.DataFrame()
xai_missing_manifest_df = pd.DataFrame()
xai_base_overlap_df = pd.DataFrame()
missing_df = pd.DataFrame()
seed_metrics_df = pd.DataFrame()
metrics_df = pd.DataFrame()
seed_grouped_pfi_df = pd.DataFrame()
grouped_pfi_df = pd.DataFrame()
seed_grouped_shap_df = pd.DataFrame()
grouped_shap_df = pd.DataFrame()
group_share_summary_df = pd.DataFrame()
group_share_overview_df = pd.DataFrame()
seed_agreement_df = pd.DataFrame()
agreement_df = pd.DataFrame()
stability_summary_df = pd.DataFrame()
stability_by_group_df = pd.DataFrame()
stability_overview_df = pd.DataFrame()
seed_mode_delta_df = pd.DataFrame()
mode_delta_df = pd.DataFrame()
mode_transition_summary_df = pd.DataFrame()
primary_transition_df = pd.DataFrame()
supporting_transition_df = pd.DataFrame()
rq1_summary_df = pd.DataFrame()
rq2_summary_df = pd.DataFrame()
selected_cases_df = pd.DataFrame()
local_figure_df = pd.DataFrame()
scenario_results_df = pd.DataFrame()
scenario_summary_df = pd.DataFrame()
rq3_summary_df = pd.DataFrame()
run_log_df = pd.DataFrame()
smoke_run_log_df = pd.DataFrame()
artifact_status_df = pd.DataFrame()
coverage_export_df = pd.DataFrame()
missing_export_df = pd.DataFrame()
coverage_summary_df = pd.DataFrame()
scope_validation_df = pd.DataFrame()
notebook_validation_df = pd.DataFrame()
mode_definition_df = pd.DataFrame()
weather_role_df = pd.DataFrame()
mode_completion_df = pd.DataFrame()
mode_weather_completion_df = pd.DataFrame()
thesis_packets = {}
ACTIVE_XAI_MODES = MODES
plot_manifest_rows = []
plot_manifest_df = pd.DataFrame()
PLOT_MANIFEST_COLUMNS = [
    'plot_type',
    'path',
    'model_family',
    'regime',
    'building',
    'mode',
    'weather_mode',
    'horizon_h',
    'case_type',
    'feature_name',
    'thesis_role',
    'analysis_theme',
]


def register_plot(path: Path, *, plot_type: str, **meta: object) -> None:
    row = {col: pd.NA for col in PLOT_MANIFEST_COLUMNS}
    row.update({'plot_type': plot_type, 'path': str(path)})
    row.update(meta)
    plot_manifest_rows.append(row)


def rebuild_detail_cache(
        existing_cache: dict[tuple[str, str, str, str, str, int, int], dict[str, object]] | None,
        detail_keys: set[tuple[str, str, str, str, str, int, int]],
        *,
        label: str,
    ) -> dict[tuple[str, str, str, str, str, int, int], dict[str, object]]:
        recovered = existing_cache.copy() if existing_cache else {}
        if not detail_keys:
            return recovered

        if seed_metrics_df.empty:
            display(Markdown(f'No saved seed metrics are available, so detail-cache recovery for {label} cannot run.'))
            return recovered

        available_keys = {
            (
                str(model_family),
                str(regime),
                str(building),
                str(mode),
                str(weather_mode),
                int(horizon_h),
                int(seed),
            )
            for model_family, regime, building, mode, weather_mode, horizon_h, seed in seed_metrics_df.loc[
                :, ['model_family', 'regime', 'building', 'mode', 'weather_mode', 'horizon_h', 'seed']
            ].drop_duplicates().itertuples(index=False, name=None)
        }
        requested_keys = {tuple(key) for key in detail_keys}
        missing_rows = []
        for key in sorted(requested_keys):
            if key not in available_keys:
                missing_rows.append(
                    {
                        'model_family': key[0],
                        'regime': key[1],
                        'building': key[2],
                        'mode': key[3],
                        'weather_mode': key[4],
                        'horizon_h': key[5],
                        'seed': key[6],
                        'reason': 'No saved seed metrics exist for this slice.',
                    }
                )
        available_detail_keys = requested_keys & available_keys
        if missing_rows:
            display(Markdown(f'Detail-cache recovery skipped {len(missing_rows)} rows for {label}.'))
            display(pd.DataFrame(missing_rows))
        return {key: value for key, value in recovered.items() if key in available_detail_keys}



## What This Notebook Answers

This notebook keeps the thesis XAI story centered on four questions:

1. Which driver classes matter more as the cumulative horizon grows?
2. What changes when historical weather memory, system dynamics, and static context are added across `M0-M4`?
3. How does that story change between `FW0`, `FW2`, and `FW1`?
4. Are the grouped explanations stable and internally consistent enough to support bounded thesis claims?

The notebook is reload-first by default. New XAI runs are triggered manually only when the underlying model-family results needed for a mode are actually available. The broad thesis rollup remains the headline language, and the fine grouped taxonomy is layered underneath it wherever a more granular diagnostic read is needed.



## How To Read Modes And Weather Regimes

The feature modes and weather regimes are part of the thesis design, not ad hoc notebook filters. The notebook always keeps the full `M0-M4` grammar visible even when some modes are still pending in the current saved outputs.

`M0-M4` is the experiment grammar, not the XAI taxonomy. The modes tell the notebook which feature blocks were available to the model, while the grouped-XAI taxonomy tells the notebook how those features should be interpreted after fitting.

Read the main mode transitions as follows:
- `M1 - M0`: added historical weather memory
- `M2 - M0`: added system dynamics
- `M3 - M2`: added historical weather memory on top of system dynamics
- `M4 - M3`: added static building context

`FW0-FW2` is the weather-information regime grammar, not a feature family. It controls whether future weather is absent, forecast-like, or oracle, and should therefore be read separately from the mode grammar.

Read the weather regimes as follows:
- `FW0`: clean inertia identification without future-weather confounding
- `FW2`: realistic forecast-like future weather with noise
- `FW1`: oracle upper bound, useful for comparison but not operationally realistic


In [3]:

base_frames = mfc.build_base_frame_cache(config)

desired_full_manifest_df = xns.build_xai_matrix_manifest(
    model_families=MODEL_FAMILIES,
    regimes=REGIMES,
    buildings=BUILDINGS,
    modes=REQUESTED_MODES,
    weather_modes=REQUESTED_WEATHER_MODES,
    horizons=HORIZON_GRID,
    target_kind=TARGET_KIND,
    seed_list=SEED_LIST,
)
resume_basis_outputs = xns.load_saved_xai_outputs(
    NOTEBOOK_LOAD_RESULTS_DIR,
    scope_to_manifest=False,
    allowed_model_families=MODEL_FAMILIES,
    use_latest_run_log=True,
)
current_manifest_df = xns.build_completed_xai_manifest_from_outputs(
    resume_basis_outputs,
    desired_full_manifest_df,
    buildings=BUILDINGS,
    require_fine_groups=REQUIRE_FINE_GROUPS_FOR_RESUME,
    allowed_model_families=MODEL_FAMILIES,
)

xai_base_manifest_path = XAI_REPORT_BASE_DIR / 'xai_matrix_manifest.csv'
xai_base_manifest_df = pd.read_csv(xai_base_manifest_path) if xai_base_manifest_path.exists() else pd.DataFrame()
xai_base_overlap_df = xns.xai_manifest_overlap_report(
    xai_base_manifest_df,
    current_manifest_df,
    subset_label=str(XAI_REPORT_BASE_DIR),
    superset_label=str(NOTEBOOK_LOAD_RESULTS_DIR),
)
xai_inventory_df = xns.build_xai_matrix_inventory(
    desired_full_manifest_df,
    current_manifest_df,
    source_dir=NOTEBOOK_LOAD_RESULTS_DIR,
    notebook='13',
    layer='xai_matrix',
)
xai_missing_manifest_df = xns.build_missing_xai_resume_manifest(
    resume_basis_outputs,
    desired_full_manifest_df,
    buildings=BUILDINGS,
    require_fine_groups=REQUIRE_FINE_GROUPS_FOR_RESUME,
    allowed_model_families=MODEL_FAMILIES,
)
resume_scope_summary_df = pd.DataFrame(
    [
        {'scope': 'completed_now', 'manifest_rows': int(len(current_manifest_df))},
        {'scope': 'missing_now', 'manifest_rows': int(len(xai_missing_manifest_df))},
        {'scope': 'desired_full', 'manifest_rows': int(len(desired_full_manifest_df))},
    ]
)
xai_inventory_summary_df = (
    xai_inventory_df.groupby(['status', 'mode', 'regime', 'weather_mode'], as_index=False)
    .agg(rows=('status', 'size'))
    .sort_values(['status', 'mode', 'regime', 'weather_mode'])
    .reset_index(drop=True)
)

comparison_coverage_df = (
    comparison_metrics_df.groupby(['model_family', 'regime', 'weather_mode'], as_index=False)
    .agg(
        n_rows=('horizon_h', 'size'),
        buildings=('building', lambda s: ', '.join(sorted(pd.unique(s)))),
        modes=('mode', lambda s: ', '.join(sorted(pd.unique(s)))),
        min_h=('horizon_h', 'min'),
        max_h=('horizon_h', 'max'),
    )
    .sort_values(['model_family', 'regime', 'weather_mode'])
    .reset_index(drop=True)
)

broad_feature_group_tables = {
    mode: xns.feature_group_table(
        xns.make_feature_groups(
            xns.mode_temporal_features(mode) + xns.mode_static_features(mode),
            taxonomy='broad',
        ),
        taxonomy='broad',
    )
    for mode in REQUESTED_MODES
}
fine_feature_group_tables = {
    mode: xns.feature_group_table(
        xns.make_feature_groups(
            xns.mode_temporal_features(mode) + xns.mode_static_features(mode),
            taxonomy='fine',
        ),
        taxonomy='fine',
        include_rollup=True,
    )
    for mode in REQUESTED_MODES
}

preview_building = ANCHOR_BUILDINGS[0]
preview_horizon = DETAIL_HORIZONS[0]
mode_definition_df = xns.build_thesis_mode_definition_table()
weather_role_df = xns.build_thesis_weather_role_table()
broad_taxonomy_definition_df = xns.build_broad_taxonomy_definition_table()
fine_taxonomy_definition_df = xns.build_fine_taxonomy_definition_table()
weather_mode_semantics_df = mfc.build_weather_mode_semantics_table(
    config,
    base_frames={preview_building: base_frames[preview_building]},
    sample_building=preview_building,
    sample_horizon_h=preview_horizon,
)
comparison_mode_status_df = pd.DataFrame(
    [
        {
            'mode': mode,
            'present_in_current_comparison_results': bool(mode in AVAILABLE_COMPARISON_MODES),
        }
        for mode in REQUESTED_MODES
    ]
)
preview_frame = base_frames[preview_building]['setB'].loc[:, [
    col for col in [
        'datetime',
        'heat_kwh',
        'feat_heat_obs',
        'feat_outdoor_temp_c',
        'feat_temp_roll24h',
        'feat_space_deltaT_c',
        'feat_return_deltaT_c',
    ]
    if col in base_frames[preview_building]['setB'].columns
]].head(12)

display(Markdown('### Current comparison coverage inherited from notebook 12'))
display(comparison_coverage_df)
display(Markdown('### Full thesis mode grammar (`M0-M4`)'))
display(mode_definition_df)
display(Markdown('### Weather-regime roles in the thesis story'))
display(weather_role_df)
display(Markdown('### Broad thesis rollup taxonomy'))
display(broad_taxonomy_definition_df)
display(Markdown('### Fine diagnostic taxonomy'))
display(fine_taxonomy_definition_df)
display(Markdown('### Current comparison-matrix availability'))
display(comparison_mode_status_df)
display(Markdown('### Canonical weather-mode construction preview'))
display(weather_mode_semantics_df)
display(Markdown('### XAI base vs current overlap check'))
display(xai_base_overlap_df)
display(Markdown('### XAI inventory summary'))
display(xai_inventory_summary_df)
display(Markdown('### Resume scope summary'))
display(resume_scope_summary_df)
if not xai_missing_manifest_df.empty:
    display(Markdown('### Missing XAI manifest preview'))
    display(xai_missing_manifest_df.head(12))
display(Markdown('### Feature preview'))
display(preview_frame)
for mode in REQUESTED_MODES:
    display(Markdown(f'### Broad rollup feature groups for `{mode}`'))
    display(broad_feature_group_tables[mode])
    display(Markdown(f'### Fine diagnostic feature groups for `{mode}`'))
    display(fine_feature_group_tables[mode])


/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/xai_notebook_support.py:2426: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  missing = missing.loc[missing["_present"].fillna(False) == False].drop(columns="_present")
/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/xai_notebook_support.py:2526: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  inventory["completed_now"] = inventory["completed_now"].fillna(False).astype(bool)
/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/xai_notebook_support.py:2426: FutureWar

### Current comparison coverage inherited from notebook 12

,model_family,regime,weather_mode,n_rows,buildings,modes,min_h,max_h
0,lstm,per_building,FW0,300,"LIB, SOC, U02B, U03, U05, U06","M0, M1, M2, M3, M4",1,36
1,lstm,per_building,FW1,300,"LIB, SOC, U02B, U03, U05, U06","M0, M1, M2, M3, M4",1,36
2,lstm,per_building,FW2,300,"LIB, SOC, U02B, U03, U05, U06","M0, M1, M2, M3, M4",1,36
3,lstm,pooled_same_buildings,FW0,300,"LIB, SOC, U02B, U03, U05, U06","M0, M1, M2, M3, M4",1,36
4,lstm,pooled_same_buildings,FW1,300,"LIB, SOC, U02B, U03, U05, U06","M0, M1, M2, M3, M4",1,36
5,lstm,pooled_same_buildings,FW2,300,"LIB, SOC, U02B, U03, U05, U06","M0, M1, M2, M3, M4",1,36
6,xgboost,per_building,FW0,300,"LIB, SOC, U02B, U03, U05, U06","M0, M1, M2, M3, M4",1,36
7,xgboost,per_building,FW1,300,"LIB, SOC, U02B, U03, U05, U06","M0, M1, M2, M3, M4",1,36
8,xgboost,per_building,FW2,300,"LIB, SOC, U02B, U03, U05, U06","M0, M1, M2, M3, M4",1,36
9,xgboost,pooled_same_buildings,FW0,300,"LIB, SOC, U02B, U03, U05, U06","M0, M1, M2, M3, M4",1,36


### Full thesis mode grammar (`M0-M4`)

,mode,feature_increment,thesis_reading
0,M0,Lean temporal core,"Reference mode with recent demand, current wea..."
1,M1,Add historical weather memory,Tests whether lagged weather-memory features h...
2,M2,Add system dynamics,Tests whether subsystem delta-T and activity p...
3,M3,Add historical weather memory on top of M2,Separates the incremental value of historical ...
4,M4,Add static building context on top of M3,Tests whether static building descriptors add ...


...

,feature_group,feature_group_label,n_features,features,broad_rollup,broad_rollup_label
0,demand_signal,Demand signal,1,feat_heat_obs,recent_demand_history,Recent demand history
1,current_weather,Current weather,3,"feat_outdoor_temp_c, feat_wind_ms, feat_solar_...",current_weather,Current weather
2,weather_memory,Weather memory,1,feat_temp_roll24h,historical_weather_memory,Historical weather memory
3,space_heating_dynamics,Space-heating dynamics,3,"feat_space_heat_active, feat_space_deltaT_c, f...",system_dynamic_proxies,System dynamics
4,ventilation_dynamics,Ventilation and DHW dynamics,3,"feat_vent_heat_active, feat_vent_deltaT_c, fea...",system_dynamic_proxies,System dynamics
5,calendar,Calendar,4,"feat_hour_sin, feat_hour_cos, feat_dow_sin, fe...",calendar,Calendar


### Broad rollup feature groups for `M4`

,feature_group,feature_group_label,n_features,features
0,recent_demand_history,Recent demand history,1,feat_heat_obs
1,historical_weather_memory,Historical weather memory,1,feat_temp_roll24h
2,current_weather,Current weather,3,"feat_outdoor_temp_c, feat_wind_ms, feat_solar_..."
3,system_dynamic_proxies,System dynamics,6,"feat_space_heat_active, feat_space_deltaT_c, f..."
4,calendar,Calendar,4,"feat_hour_sin, feat_hour_cos, feat_dow_sin, fe..."
5,static_building_context,Static building context,20,"stat_heated_area_m2, stat_usage_non_res_share_..."


### Fine diagnostic feature groups for `M4`

,feature_group,feature_group_label,n_features,features,broad_rollup,broad_rollup_label
0,demand_signal,Demand signal,1,feat_heat_obs,recent_demand_history,Recent demand history
1,current_weather,Current weather,3,"feat_outdoor_temp_c, feat_wind_ms, feat_solar_...",current_weather,Current weather
2,weather_memory,Weather memory,1,feat_temp_roll24h,historical_weather_memory,Historical weather memory
3,space_heating_dynamics,Space-heating dynamics,3,"feat_space_heat_active, feat_space_deltaT_c, f...",system_dynamic_proxies,System dynamics
4,ventilation_dynamics,Ventilation and DHW dynamics,3,"feat_vent_heat_active, feat_vent_deltaT_c, fea...",system_dynamic_proxies,System dynamics
5,calendar,Calendar,4,"feat_hour_sin, feat_hour_cos, feat_dow_sin, fe...",calendar,Calendar
6,static_profile_and_inventory,Static profile and inventory,14,"stat_heated_area_m2, stat_usage_non_res_share_...",static_building_context,Static building context
7,static_ehr_morphology,Static EHR morphology,6,"ehr_compactness_ratio, ehr_max_floors, ehr_vol...",static_building_context,Static building context


## Quick Smoke Check

Run this cheap preflight before any heavier XAI cells. It checks the current kernel, saved artifact presence, and the active notebook export directory without running the notebook itself.


In [4]:
smoke_rows = [
    {'check': 'kernel_python', 'status': 'ok' if 'thesis-project/.venv' in sys.executable else 'warn', 'detail': sys.executable},
    {'check': 'comparison_results_dir_exists', 'status': 'ok' if COMPARISON_RESULTS_DIR.exists() else 'block', 'detail': str(COMPARISON_RESULTS_DIR)},
    {'check': 'load_results_dir_exists', 'status': 'ok' if NOTEBOOK_LOAD_RESULTS_DIR.exists() else 'block', 'detail': str(NOTEBOOK_LOAD_RESULTS_DIR)},
    {'check': 'report_export_dir_exists', 'status': 'ok' if NOTEBOOK_REPORT_EXPORT_DIR.exists() else 'warn', 'detail': str(NOTEBOOK_REPORT_EXPORT_DIR)},
]
for name in ['xai_seed_metrics.csv', 'xai_seed_grouped_pfi.csv', 'xai_seed_grouped_shap.csv', 'xai_run_log.csv']:
    path = NOTEBOOK_LOAD_RESULTS_DIR / name
    smoke_rows.append({'check': name, 'status': 'ok' if path.exists() else 'block', 'detail': str(path)})
scope_validation_path = NOTEBOOK_REPORT_EXPORT_DIR / 'xai_scope_validation.csv'
if scope_validation_path.exists():
    scope_validation_probe_df = pd.read_csv(scope_validation_path)
    smoke_rows.append({
        'check': 'scope_validation_all_valid',
        'status': 'ok' if scope_validation_probe_df.get('valid', pd.Series(dtype=bool)).fillna(False).all() else 'warn',
        'detail': f'rows={len(scope_validation_probe_df)} path={scope_validation_path}',
    })
smoke_df = pd.DataFrame(smoke_rows)
display(smoke_df)


,check,status,detail
0,kernel_python,ok,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
1,comparison_results_dir_exists,ok,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
2,load_results_dir_exists,ok,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
3,report_export_dir_exists,ok,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
4,xai_seed_metrics.csv,ok,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
5,xai_seed_grouped_pfi.csv,ok,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
6,xai_seed_grouped_shap.csv,ok,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
7,xai_run_log.csv,ok,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
8,scope_validation_all_valid,warn,rows=8 path=/Users/mihkeluutar/Documents/TalTe...


## Manual Full-Matrix Trigger

The notebook stays reload-first by default. In the current backfill profile, the execution cell now builds its run scope from the actually missing fine grouped-XAI outputs on disk, so it should schedule only the backfill slots instead of presenting the whole thesis matrix as fresh work.


In [5]:
detail_requests = {
    (model_family, 'per_building', building, mode, FOCUS_WEATHER_MODE, horizon_h, DETAIL_SEED)
    for model_family in RUN_MODEL_FAMILIES
    for building in ANCHOR_BUILDINGS
    for mode in RUN_MODES
    for horizon_h in DETAIL_HORIZONS
}

missing_run_modes = sorted(set(RUN_MODES) - set(AVAILABLE_COMPARISON_MODES))

if not RUN_FULL_MATRIX_IN_NOTEBOOK:
    display(Markdown(
        'Full XAI matrix execution is disabled by default so the notebook stays scoped to saved CSV artifacts. '
        'When you want to backfill missing fine grouped-XAI artifacts across the full matrix, switch `XAI_WORKFLOW_PROFILE` to `resume_fine_backfill_all_modes`, rerun the setup cells, and then run this cell.'
    ))
elif missing_run_modes:
    raise RuntimeError(
        'The current comparison results do not yet contain the model-family slices needed for these requested thesis modes: '
        + ', '.join(missing_run_modes)
        + '. Complete the upstream comparison exports first, or narrow `RUN_MODES` manually for this notebook run.'
    )
else:
    if REQUIRE_SMOKE_PASS:
        if RUN_MATRIX_SMOKE and 'smoke_missing_df' not in globals():
            raise RuntimeError('Run the smoke-test cell first when `REQUIRE_SMOKE_PASS = True`.')
        if RUN_MATRIX_SMOKE and not smoke_missing_df.empty:
            raise RuntimeError('Smoke test has missing rows. Fix them before launching the full matrix.')

    full_run_manifest_df = (
        xai_missing_manifest_df.copy()
        if MANIFEST_STRATEGY == 'missing_only'
        else desired_full_manifest_df.copy()
    )

    if full_run_manifest_df.empty:
        display(Markdown('All requested fine grouped-XAI slots already exist for the current resume scope, so no backfill run is needed.'))
    else:
        display(Markdown(f'Running the XAI backfill over **{len(full_run_manifest_df)}** manifest rows that are still missing under the active fine-group resume rules.'))

        run_config = xns.clone_config_with_overrides(
        config,
        buildings=RUN_BUILDINGS,
        horizons=RUN_HORIZONS,
        regimes=RUN_REGIMES,
        weather_modes=RUN_WEATHER_MODES,
        modes=RUN_MODES,
        model_families=RUN_MODEL_FAMILIES,
        random_seed=DETAIL_SEED,
        results_dir=FULL_MATRIX_RESULTS_DIR,
    )
        xns.print_xai_resume_diagnostics(
            run_config.results_dir,
            manifest_df=full_run_manifest_df,
            buildings=RUN_BUILDINGS,
            require_fine_groups=REQUIRE_FINE_GROUPS_FOR_RESUME,
        )

        matrix_artifacts = xns.run_broad_xai_matrix(
            run_config,
            base_frames,
            manifest_df=full_run_manifest_df,
            regimes=RUN_REGIMES,
            buildings=RUN_BUILDINGS,
            modes=RUN_MODES,
            weather_modes=RUN_WEATHER_MODES,
            horizons_by_target_kind={TARGET_KIND: RUN_HORIZONS},
            target_kind=TARGET_KIND,
            seed_list=RUN_SEED_LIST,
            model_families=RUN_MODEL_FAMILIES,
            pfi_repeats_by_family={family: PFI_REPEATS_BY_FAMILY[family] for family in RUN_MODEL_FAMILIES},
            shap_background_size=SHAP_BACKGROUND_SIZE,
            shap_explain_size=SHAP_EXPLAIN_SIZE,
            lstm_pfi_max_rows=LSTM_PFI_MAX_ROWS,
            detail_requests=detail_requests,
            save_artifacts=True,
            resume_existing=RESUME_EXISTING,
            save_after_each_slot=SAVE_AFTER_EACH_SLOT,
            save_every_n_slots=SAVE_EVERY_N_SLOTS,
            continue_on_error=CONTINUE_ON_ERROR,
            verbose=True,
            require_fine_groups_for_resume=REQUIRE_FINE_GROUPS_FOR_RESUME,
        )
        matrix_artifacts['manifest_df'] = desired_full_manifest_df.copy()
        desired_full_manifest_df.to_csv(FULL_MATRIX_RESULTS_DIR / 'xai_matrix_manifest.csv', index=False)

        display(Markdown('### Full matrix run completed'))
        if not matrix_artifacts['run_log_df'].empty:
            display(matrix_artifacts['run_log_df'].tail(12))


Running the XAI backfill over **3816** manifest rows that are still missing under the active fine-group resume rules.

--- xai resume diagnostics ---
cwd: /Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project
results_dir (resolved): /Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/results/xai_matrix_work_20260405
  xai_matrix_manifest.csv: exists=True data_rows=4860
  xai_seed_metrics.csv: exists=True data_rows=4860
  xai_seed_grouped_pfi.csv: exists=True data_rows=24624
  xai_seed_grouped_shap.csv: exists=True data_rows=24624
  xai_seed_pfi_shap_agreement.csv: exists=True data_rows=23004
  xai_run_log.csv: exists=True data_rows=4932
combo slots complete: 174/636
first incomplete slot: family=xgboost regime=per_building mode=M0 weather=FW0 h=12 seed=42
--- end xai resume diagnostics ---
[  1/636] family=xgboost regime=per_building mode=M0 weather=FW0 h=12 seed=42
[  2/636] family=xgboost regime=per_building mode=M0 weather=FW0 h=12 seed=52
[  3/636] family=xgboost regime=per_building mode=M0 weather=FW0 h=12 seed=62
[  4/636] family=xgboost regime=per_building mode=M0 weather=FW0 h=16 se

/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:2066: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work_df["is_heating_eval"] = heating_mask(work_df).to_numpy(dtype=bool)
/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:2066: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work_df["is_heating_eval"] = heating_mask(work_df).to_numpy(dtype=bool)
/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:2066: Perf

[284/636] family=xgboost regime=per_building mode=M4 weather=FW1 h=36 seed=52


/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:2066: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work_df["is_heating_eval"] = heating_mask(work_df).to_numpy(dtype=bool)
/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:2066: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work_df["is_heating_eval"] = heating_mask(work_df).to_numpy(dtype=bool)
/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:2066: Perf

...

/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:2066: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work_df["is_heating_eval"] = heating_mask(work_df).to_numpy(dtype=bool)
/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:2066: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work_df["is_heating_eval"] = heating_mask(work_df).to_numpy(dtype=bool)
/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:2066: Perf

[636/636] family=xgboost regime=pooled_same_buildings mode=M4 weather=FW2 h=36 seed=62


/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:2066: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work_df["is_heating_eval"] = heating_mask(work_df).to_numpy(dtype=bool)
/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:2066: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work_df["is_heating_eval"] = heating_mask(work_df).to_numpy(dtype=bool)
/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:2066: Perf

### Full matrix run completed

,model_family,regime,building,mode,weather_mode,horizon_h,target_kind,seed,training_scope,slot_index,status,elapsed_s,error_type,error_message
8736,xgboost,pooled_same_buildings,LIB,M4,FW1,36,cum,52,POOLED,647,ok,12.318461,NaN,NaN
8737,xgboost,pooled_same_buildings,SOC,M4,FW1,36,cum,52,POOLED,647,ok,14.437627,NaN,NaN
8738,xgboost,pooled_same_buildings,U02B,M4,FW1,36,cum,52,POOLED,647,ok,16.469501,NaN,NaN
8739,xgboost,pooled_same_buildings,U03,M4,FW1,36,cum,52,POOLED,647,ok,18.557993,NaN,NaN
8740,xgboost,pooled_same_buildings,U05,M4,FW1,36,cum,52,POOLED,647,ok,20.691042,NaN,NaN
8741,xgboost,pooled_same_buildings,U06,M4,FW1,36,cum,52,POOLED,647,ok,22.783180,NaN,NaN
8742,xgboost,pooled_same_buildings,LIB,M4,FW1,36,cum,62,POOLED,648,ok,12.309258,NaN,NaN
8743,xgboost,pooled_same_buildings,SOC,M4,FW1,36,cum,62,POOLED,648,ok,14.401028,NaN,NaN
8744,xgboost,pooled_same_buildings,U02B,M4,FW1,36,cum,62,POOLED,648,ok,16.477169,NaN,NaN
8745,xgboost,pooled_same_buildings,U03,M4,FW1,36,cum,62,POOLED,648,ok,18.640918,NaN,NaN


## Saved Artifact Reload


In [6]:

if MERGE_SOURCE_RESULTS_DIRS:
    merge_outputs = []
    for source_dir in MERGE_SOURCE_RESULTS_DIRS:
        source_path = Path(source_dir).resolve()
        merge_outputs.append(
            xns.load_saved_xai_outputs(
                source_path,
                allowed_model_families=MODEL_FAMILIES,
                use_latest_run_log=True,
            )
        )
    merged_outputs = xns.merge_xai_outputs(*merge_outputs)
    merge_manifest_df = merged_outputs['manifest_df'].copy()
    saved_outputs = xns.scope_xai_outputs_to_manifest(
        merged_outputs,
        merge_manifest_df,
        allowed_model_families=MODEL_FAMILIES,
        use_latest_run_log=True,
    )
else:
    saved_outputs = matrix_artifacts if matrix_artifacts else xns.load_saved_xai_outputs(
        NOTEBOOK_LOAD_RESULTS_DIR,
        scope_to_manifest=True,
        allowed_model_families=MODEL_FAMILIES,
        use_latest_run_log=True,
    )

current_outputs = matrix_artifacts if matrix_artifacts and not MERGE_SOURCE_RESULTS_DIRS else saved_outputs
manifest_df = saved_outputs['manifest_df'].copy()
seed_metrics_df = saved_outputs['seed_metrics_df'].copy()
seed_grouped_pfi_df = saved_outputs['seed_grouped_pfi_df'].copy()
seed_grouped_pfi_fine_df = saved_outputs.get('seed_grouped_pfi_fine_df', pd.DataFrame()).copy()
seed_grouped_shap_df = saved_outputs['seed_grouped_shap_df'].copy()
seed_grouped_shap_fine_df = saved_outputs.get('seed_grouped_shap_fine_df', pd.DataFrame()).copy()
seed_agreement_df = saved_outputs['seed_agreement_df'].copy()
seed_agreement_fine_df = saved_outputs.get('seed_agreement_fine_df', pd.DataFrame()).copy()
run_log_df = saved_outputs['run_log_df'].copy()
artifact_status_df = xns.build_xai_artifact_status_table(NOTEBOOK_LOAD_RESULTS_DIR)

saved_shape_df = pd.DataFrame(
    [
        {'artifact': 'manifest_df', 'rows': int(len(manifest_df))},
        {'artifact': 'seed_metrics_df', 'rows': int(len(seed_metrics_df))},
        {'artifact': 'seed_grouped_pfi_df', 'rows': int(len(seed_grouped_pfi_df))},
        {'artifact': 'seed_grouped_pfi_fine_df', 'rows': int(len(seed_grouped_pfi_fine_df))},
        {'artifact': 'seed_grouped_shap_df', 'rows': int(len(seed_grouped_shap_df))},
        {'artifact': 'seed_grouped_shap_fine_df', 'rows': int(len(seed_grouped_shap_fine_df))},
        {'artifact': 'seed_agreement_df', 'rows': int(len(seed_agreement_df))},
        {'artifact': 'seed_agreement_fine_df', 'rows': int(len(seed_agreement_fine_df))},
        {'artifact': 'run_log_df', 'rows': int(len(run_log_df))},
    ]
)

display(Markdown('### Saved artifact shapes'))
display(saved_shape_df)
display(Markdown('### Saved artifact overview'))
display(artifact_status_df)
if not run_log_df.empty:
    display(Markdown('### Latest run-log rows'))
    display(run_log_df.tail(12))


### Saved artifact shapes

,artifact,rows
0,manifest_df,4860
1,seed_metrics_df,4860
2,seed_grouped_pfi_df,24624
3,seed_grouped_pfi_fine_df,31752
4,seed_grouped_shap_df,24624
5,seed_grouped_shap_fine_df,31752
6,seed_agreement_df,23004
7,seed_agreement_fine_df,31752
8,run_log_df,8748


### Saved artifact overview

,artifact,exists,path,rows
0,results,True,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,NaN
1,plots,True,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,NaN
2,xai_run_log.csv,True,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,8748.0
3,xai_seed_metrics.csv,True,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,4860.0
4,xai_metrics.csv,False,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,NaN
5,xai_predictions.csv,False,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,NaN
6,xai_seed_grouped_pfi.csv,True,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,24624.0
7,xai_grouped_pfi.csv,False,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,NaN
8,xai_seed_grouped_pfi_fine.csv,True,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,31752.0
9,xai_grouped_pfi_fine.csv,False,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,NaN


### Latest run-log rows

,model_family,regime,building,mode,weather_mode,horizon_h,target_kind,seed,training_scope,slot_index,status,elapsed_s,error_type,error_message
8736,xgboost,pooled_same_buildings,LIB,M4,FW1,36,cum,52,POOLED,647,ok,12.318461,NaN,NaN
8737,xgboost,pooled_same_buildings,SOC,M4,FW1,36,cum,52,POOLED,647,ok,14.437627,NaN,NaN
8738,xgboost,pooled_same_buildings,U02B,M4,FW1,36,cum,52,POOLED,647,ok,16.469501,NaN,NaN
8739,xgboost,pooled_same_buildings,U03,M4,FW1,36,cum,52,POOLED,647,ok,18.557993,NaN,NaN
8740,xgboost,pooled_same_buildings,U05,M4,FW1,36,cum,52,POOLED,647,ok,20.691042,NaN,NaN
8741,xgboost,pooled_same_buildings,U06,M4,FW1,36,cum,52,POOLED,647,ok,22.783180,NaN,NaN
8742,xgboost,pooled_same_buildings,LIB,M4,FW1,36,cum,62,POOLED,648,ok,12.309258,NaN,NaN
8743,xgboost,pooled_same_buildings,SOC,M4,FW1,36,cum,62,POOLED,648,ok,14.401028,NaN,NaN
8744,xgboost,pooled_same_buildings,U02B,M4,FW1,36,cum,62,POOLED,648,ok,16.477169,NaN,NaN
8745,xgboost,pooled_same_buildings,U03,M4,FW1,36,cum,62,POOLED,648,ok,18.640918,NaN,NaN


In [7]:
if AUTO_EXPORT_REPORT_READY and not seed_metrics_df.empty:
    xns.export_report_ready_xai_artifacts(
        current_outputs,
        NOTEBOOK_REPORT_EXPORT_DIR,
        manifest_df=manifest_df,
        desired_manifest_df=desired_full_manifest_df,
        allowed_model_families=MODEL_FAMILIES,
    )
    display(Markdown(f'### Report-ready notebook export refreshed at `{NOTEBOOK_REPORT_EXPORT_DIR}`'))
else:
    display(Markdown('### Report-ready notebook export skipped'))

coverage_export_path = NOTEBOOK_REPORT_EXPORT_DIR / 'xai_manifest_coverage.csv'
missing_export_path = NOTEBOOK_REPORT_EXPORT_DIR / 'xai_manifest_missing.csv'
scope_validation_path = NOTEBOOK_REPORT_EXPORT_DIR / 'xai_scope_validation.csv'
mode_transition_path = NOTEBOOK_REPORT_EXPORT_DIR / 'xai_mode_transition_summary.csv'
group_share_summary_path = NOTEBOOK_REPORT_EXPORT_DIR / 'xai_group_share_summary.csv'
def safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

coverage_export_df = safe_read_csv(coverage_export_path)
missing_export_df = safe_read_csv(missing_export_path)
scope_validation_df = safe_read_csv(scope_validation_path)
mode_transition_summary_df = safe_read_csv(mode_transition_path)
group_share_summary_df = safe_read_csv(group_share_summary_path)
if not scope_validation_df.empty:
    display(Markdown('### Current scope validation'))
    display(scope_validation_df)


### Report-ready notebook export refreshed at `/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/results/xai_matrix_work_20260405/_report_ready`

### Current scope validation

,artifact,rows,rows_outside_manifest,valid
0,run_log_df,4860,972,False
1,seed_agreement_df,23004,972,False
2,seed_agreement_fine_df,7128,972,False
3,seed_grouped_pfi_df,24624,972,False
4,seed_grouped_pfi_fine_df,7128,972,False
5,seed_grouped_shap_df,24624,972,False
6,seed_grouped_shap_fine_df,7128,972,False
7,seed_metrics_df,4860,972,False



## Current Completion Status

The notebook always keeps the full thesis grammar visible. Completion is shown explicitly so that `M1` and `M3` remain visible as thesis modes even when they are still pending in the current saved scope, rather than being dropped from the grammar.


In [8]:
loaded_modes = tuple(mode for mode in REQUESTED_MODES if mode in set(seed_metrics_df.get('mode', pd.Series(dtype=object)).dropna().astype(str)))
loaded_families = sorted(seed_metrics_df['model_family'].astype(str).unique().tolist()) if not seed_metrics_df.empty else []
ACTIVE_XAI_MODES = loaded_modes if loaded_modes else REQUESTED_MODES
mode_completion_df = xns.build_xai_mode_completion_table(
    desired_full_manifest_df,
    seed_metrics_df,
    requested_modes=REQUESTED_MODES,
)
mode_weather_completion_df = xns.build_xai_mode_weather_completion_table(
    desired_full_manifest_df,
    seed_metrics_df,
    requested_modes=REQUESTED_MODES,
    weather_modes=REQUESTED_WEATHER_MODES,
)
pending_modes = mode_completion_df.loc[mode_completion_df['status'] == 'pending', 'mode'].tolist() if not mode_completion_df.empty else []
partial_modes = mode_completion_df.loc[mode_completion_df['status'] == 'partial', 'mode'].tolist() if not mode_completion_df.empty else []
scope_validation_ok = bool(scope_validation_df['valid'].all()) if not scope_validation_df.empty and 'valid' in scope_validation_df.columns else False
manifest_missing_ok = bool(missing_export_df.empty)
overlap_ok = bool(xai_base_overlap_df['is_subset'].all()) if not xai_base_overlap_df.empty and 'is_subset' in xai_base_overlap_df.columns else False

notebook_validation_rows = [
    {'check': 'load_results_dir_exists', 'status': 'ok' if NOTEBOOK_LOAD_RESULTS_DIR.exists() else 'block', 'detail': str(NOTEBOOK_LOAD_RESULTS_DIR)},
    {'check': 'report_export_dir_exists', 'status': 'ok' if NOTEBOOK_REPORT_EXPORT_DIR.exists() else 'warn', 'detail': str(NOTEBOOK_REPORT_EXPORT_DIR)},
    {'check': 'seed_metrics_loaded', 'status': 'ok' if not seed_metrics_df.empty else 'block', 'detail': f'rows={len(seed_metrics_df)}'},
    {'check': 'model_family_scope', 'status': 'ok' if loaded_families == list(MODEL_FAMILIES) else 'warn', 'detail': f'loaded={loaded_families} expected={list(MODEL_FAMILIES)}'},
    {
        'check': 'thesis_mode_grammar_visibility',
        'status': 'ok' if not pending_modes and not partial_modes else 'warn',
        'detail': f'pending={pending_modes if pending_modes else "(none)"} partial={partial_modes if partial_modes else "(none)"}',
    },
    {'check': 'xai_base_subset_of_current_resume_dir', 'status': 'ok' if overlap_ok else 'block', 'detail': f'rows={len(xai_base_overlap_df)}'},
    {'check': 'manifest_scope_validation', 'status': 'ok' if scope_validation_ok else 'warn', 'detail': f'rows={len(scope_validation_df)}'},
    {'check': 'manifest_missing_rows', 'status': 'ok' if manifest_missing_ok else 'warn', 'detail': f'missing_rows={len(missing_export_df)}'},
    {'check': 'transition_summary_saved', 'status': 'ok' if not mode_transition_summary_df.empty else 'warn', 'detail': f'rows={len(mode_transition_summary_df)}'},
]
notebook_validation_df = pd.DataFrame(notebook_validation_rows)
display(Markdown('### Notebook validation status'))
display(notebook_validation_df)
display(Markdown('### Thesis mode completion'))
display(mode_completion_df)
display(Markdown('### Mode / weather completion grid'))
display(mode_weather_completion_df)


### Notebook validation status

,check,status,detail
0,load_results_dir_exists,ok,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
1,report_export_dir_exists,ok,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
2,seed_metrics_loaded,ok,rows=4860
3,model_family_scope,ok,loaded=['xgboost'] expected=['xgboost']
4,thesis_mode_grammar_visibility,ok,pending=(none) partial=(none)
5,xai_base_subset_of_current_resume_dir,block,rows=1
6,manifest_scope_validation,warn,rows=8
7,manifest_missing_rows,ok,missing_rows=0
8,transition_summary_saved,ok,rows=324


### Thesis mode completion

,mode,status,completed_slots,total_slots,coverage_pct,thesis_reading,detail
0,M0,complete,972,972,1.0,"Reference mode with recent demand, current wea...",972/972 manifest slots currently have saved XA...
1,M1,complete,972,972,1.0,Tests whether lagged weather-memory features h...,972/972 manifest slots currently have saved XA...
2,M2,complete,972,972,1.0,Tests whether subsystem delta-T and activity p...,972/972 manifest slots currently have saved XA...
3,M3,complete,972,972,1.0,Separates the incremental value of historical ...,972/972 manifest slots currently have saved XA...
4,M4,complete,972,972,1.0,Tests whether static building descriptors add ...,972/972 manifest slots currently have saved XA...


### Mode / weather completion grid

,regime,weather_mode,mode,status,completed_slots,total_slots,coverage_pct
0,per_building,FW0,M0,complete,162,162,1.0
1,per_building,FW0,M1,complete,162,162,1.0
2,per_building,FW0,M2,complete,162,162,1.0
3,per_building,FW0,M3,complete,162,162,1.0
4,per_building,FW0,M4,complete,162,162,1.0
5,per_building,FW2,M0,complete,162,162,1.0
6,per_building,FW2,M1,complete,162,162,1.0
7,per_building,FW2,M2,complete,162,162,1.0
8,per_building,FW2,M3,complete,162,162,1.0
9,per_building,FW2,M4,complete,162,162,1.0



## Primary Thesis Output 1: Which Driver Classes Matter Across Horizon?

This section summarizes how grouped driver classes change as the cumulative horizon increases. It is a broad cross-slice view first so that the thesis still reads in the original headline vocabulary.

### Fine Driver Decomposition

The fine taxonomy is added immediately underneath the broad rollup. It is not meant to replace the broad thesis story; it is meant to show which internal sub-blocks actually sit inside those headline driver classes.


In [9]:

metrics_df = xns.aggregate_seed_metrics(seed_metrics_df)
grouped_pfi_df = xns.aggregate_seed_grouped_pfi(seed_grouped_pfi_df)
grouped_pfi_fine_df = xns.aggregate_seed_grouped_pfi(seed_grouped_pfi_fine_df)
grouped_shap_df = xns.aggregate_seed_grouped_shap(seed_grouped_shap_df)
grouped_shap_fine_df = xns.aggregate_seed_grouped_shap(seed_grouped_shap_fine_df)
seed_agreement_df = seed_agreement_df.copy()
seed_agreement_fine_df = seed_agreement_fine_df.copy()
agreement_df = xns.aggregate_seed_agreement(seed_agreement_df)
agreement_fine_df = xns.aggregate_seed_agreement(seed_agreement_fine_df)
seed_mode_delta_df, mode_delta_df = xns.compute_mode_delta_summaries(seed_metrics_df)
mode_transition_summary_df = xns.build_xai_mode_transition_summary(mode_delta_df)
stability_summary_df, stability_by_group_df = xns.compute_stability_tables(seed_grouped_pfi_df, seed_grouped_shap_df)
stability_summary_fine_df, stability_by_group_fine_df = xns.compute_stability_tables(seed_grouped_pfi_fine_df, seed_grouped_shap_fine_df)
rq1_summary_df = xns.build_rq1_accuracy_summary(metrics_df, mode_delta_df)
rq2_summary_df = xns.build_rq2_xai_stability_summary(grouped_pfi_df, grouped_shap_df, agreement_df, stability_summary_df)
rq2_summary_fine_df = xns.build_rq2_xai_stability_summary(grouped_pfi_fine_df, grouped_shap_fine_df, agreement_fine_df, stability_summary_fine_df)
coverage_summary_df = xns.build_manifest_coverage_summary(manifest_df, seed_metrics_df)
thesis_packets = xns.build_xai_thesis_packets(
    manifest_df=manifest_df,
    seed_metrics_df=seed_metrics_df,
    grouped_pfi_df=grouped_pfi_df,
    grouped_shap_df=grouped_shap_df,
    grouped_pfi_fine_df=grouped_pfi_fine_df,
    grouped_shap_fine_df=grouped_shap_fine_df,
    mode_transition_summary_df=mode_transition_summary_df,
    rq2_summary_df=rq2_summary_df,
    rq2_summary_fine_df=rq2_summary_fine_df,
    requested_modes=REQUESTED_MODES,
    weather_modes=REQUESTED_WEATHER_MODES,
)
mode_definition_df = thesis_packets['mode_definition_df'].copy()
weather_role_df = thesis_packets['weather_role_df'].copy()
broad_taxonomy_definition_df = thesis_packets['broad_taxonomy_definition_df'].copy()
fine_taxonomy_definition_df = thesis_packets['fine_taxonomy_definition_df'].copy()
mode_completion_df = thesis_packets['mode_completion_df'].copy()
mode_weather_completion_df = thesis_packets['mode_weather_completion_df'].copy()
group_share_summary_df = thesis_packets['group_share_summary_df'].copy()
group_share_overview_df = thesis_packets['group_share_overview_df'].copy()
group_share_summary_fine_df = thesis_packets['group_share_summary_fine_df'].copy()
group_share_overview_fine_df = thesis_packets['group_share_overview_fine_df'].copy()
primary_transition_df = thesis_packets['primary_transition_df'].copy()
supporting_transition_df = thesis_packets['supporting_transition_df'].copy()
stability_overview_df = thesis_packets['stability_overview_df'].copy()
stability_overview_fine_df = thesis_packets['stability_overview_fine_df'].copy()
ACTIVE_XAI_MODES = tuple(mode for mode in REQUESTED_MODES if mode in set(group_share_overview_df.get('mode', pd.Series(dtype=object)).dropna().astype(str))) or ACTIVE_XAI_MODES

full_manifest_df, missing_df = xns.validate_matrix_completeness(manifest_df, seed_metrics_df)
completed_manifest_rows = int(len(full_manifest_df) - len(missing_df))
manifest_coverage = completed_manifest_rows / len(full_manifest_df) if len(full_manifest_df) else 1.0

metrics_df.to_csv(artifact_paths['metrics'], index=False)
grouped_pfi_df.to_csv(artifact_paths['grouped_pfi'], index=False)
grouped_pfi_fine_df.to_csv(artifact_paths['grouped_pfi_fine'], index=False)
grouped_shap_df.to_csv(artifact_paths['grouped_shap'], index=False)
grouped_shap_fine_df.to_csv(artifact_paths['grouped_shap_fine'], index=False)
group_share_summary_df.to_csv(artifact_paths['group_share_summary'], index=False)
group_share_summary_fine_df.to_csv(artifact_paths['group_share_summary_fine'], index=False)
agreement_df.to_csv(artifact_paths['agreement'], index=False)
agreement_fine_df.to_csv(artifact_paths['agreement_fine'], index=False)
stability_summary_df.to_csv(artifact_paths['stability_summary'], index=False)
stability_by_group_df.to_csv(artifact_paths['stability_by_group'], index=False)
stability_summary_fine_df.to_csv(artifact_paths['stability_summary_fine'], index=False)
stability_by_group_fine_df.to_csv(artifact_paths['stability_by_group_fine'], index=False)
seed_mode_delta_df.to_csv(artifact_paths['seed_mode_deltas'], index=False)
mode_delta_df.to_csv(artifact_paths['mode_deltas'], index=False)
mode_transition_summary_df.to_csv(artifact_paths['results'] / 'xai_mode_transition_summary.csv', index=False)
rq1_summary_df.to_csv(artifact_paths['rq1_summary'], index=False)
rq2_summary_df.to_csv(artifact_paths['rq2_summary'], index=False)
rq2_summary_fine_df.to_csv(artifact_paths['rq2_summary_fine'], index=False)
full_manifest_df.to_csv(artifact_paths['results'] / 'xai_manifest_coverage.csv', index=False)
missing_df.to_csv(artifact_paths['results'] / 'xai_manifest_missing.csv', index=False)
run_log_df.to_csv(artifact_paths['results'] / 'xai_run_log_latest.csv', index=False)

scope_key_cols = xns.MATRIX_KEY_COLS + ['seed', 'training_scope']
allowed_scope_df = manifest_df.loc[:, scope_key_cols].drop_duplicates().assign(_allowed=True)
scope_validation_rows = []
for artifact_name, frame in (
    ('seed_metrics_df', seed_metrics_df),
    ('seed_grouped_pfi_df', seed_grouped_pfi_df),
    ('seed_grouped_pfi_fine_df', seed_grouped_pfi_fine_df),
    ('seed_grouped_shap_df', seed_grouped_shap_df),
    ('seed_grouped_shap_fine_df', seed_grouped_shap_fine_df),
    ('seed_agreement_df', seed_agreement_df),
    ('seed_agreement_fine_df', seed_agreement_fine_df),
    ('run_log_df', run_log_df),
):
    if frame.empty:
        scope_validation_rows.append({'artifact': artifact_name, 'rows': 0, 'rows_outside_manifest': 0, 'valid': True})
        continue
    merged = frame.loc[:, [col for col in scope_key_cols if col in frame.columns]].drop_duplicates().merge(
        allowed_scope_df,
        on=[col for col in scope_key_cols if col in frame.columns],
        how='left',
    )
    outside_count = int(merged['_allowed'].isna().sum())
    scope_validation_rows.append(
        {
            'artifact': artifact_name,
            'rows': int(len(frame)),
            'rows_outside_manifest': outside_count,
            'valid': bool(outside_count == 0),
        }
    )
scope_validation_df = pd.DataFrame(scope_validation_rows).sort_values('artifact').reset_index(drop=True)
scope_validation_df.to_csv(artifact_paths['results'] / 'xai_scope_validation.csv', index=False)
coverage_export_df = full_manifest_df.copy()
missing_export_df = missing_df.copy()

xgboost_story_modes = tuple(mode for mode in REQUESTED_MODES if mode in set(group_share_overview_df.loc[group_share_overview_df['model_family'] == 'xgboost', 'mode'].dropna().astype(str)))
broad_feature_groups = tuple(
    feature_group
    for feature_group in xns.THESIS_BROAD_FEATURE_GROUP_ORDER
    if feature_group in set(group_share_overview_df.get('feature_group', pd.Series(dtype=object)).dropna().astype(str))
)
fine_feature_groups = tuple(
    feature_group
    for feature_group in xns.THESIS_FINE_FEATURE_GROUP_ORDER
    if feature_group in set(group_share_overview_fine_df.get('feature_group', pd.Series(dtype=object)).dropna().astype(str))
)
driver_plot_rows = []
fine_driver_plot_rows = []
if not group_share_overview_df.empty and xgboost_story_modes:
    for method in ('pfi', 'shap'):
        save_path = artifact_paths['plots'] / f'thesis_driver_classes_{method}.png'
        fig = xns.plot_feature_share_horizon_lines(
            group_share_overview_df,
            method=method,
            model_family='xgboost',
            modes=xgboost_story_modes,
            feature_groups=broad_feature_groups,
            taxonomy='broad',
            title=f'Driver-class share vs horizon | XGBoost | {method.upper()} | current notebook coverage',
            save_path=save_path,
        )
        plt.close(fig)
        register_plot(
            save_path,
            plot_type=f'{method}_driver_classes',
            model_family='xgboost',
            regime='ALL',
            building='ALL',
            mode='ALL_CURRENT',
            weather_mode='ALL_CURRENT',
            horizon_h=pd.NA,
            thesis_role='primary',
            analysis_theme='inertia',
        )
        driver_plot_rows.append({'taxonomy': 'broad', 'method': method, 'path': str(save_path)})

if not group_share_overview_fine_df.empty:
    for method in ('pfi', 'shap'):
        fine_slice = group_share_overview_fine_df.loc[
            (group_share_overview_fine_df['model_family'] == 'xgboost')
            & (group_share_overview_fine_df['regime'] == 'per_building')
            & (group_share_overview_fine_df['method'] == method)
        ].copy()
        if fine_slice.empty:
            continue
        save_path = artifact_paths['plots'] / f'thesis_driver_classes_fine_{method}.png'
        fig = xns.plot_horizon_mode_group_heatmap(
            fine_slice,
            value_col='mean_share',
            taxonomy='fine',
            title=f'Fine driver decomposition | XGBoost | per_building | {method.upper()}',
            save_path=save_path,
        )
        plt.close(fig)
        register_plot(
            save_path,
            plot_type=f'{method}_driver_classes_fine',
            model_family='xgboost',
            regime='per_building',
            building='ALL',
            mode='ALL_CURRENT',
            weather_mode='ALL_CURRENT',
            horizon_h=pd.NA,
            thesis_role='primary',
            analysis_theme='inertia_fine',
        )
        fine_driver_plot_rows.append({'taxonomy': 'fine', 'method': method, 'path': str(save_path)})

integrity_df = pd.DataFrame(
    [
        {'artifact': 'manifest_rows', 'value': int(len(manifest_df))},
        {'artifact': 'seed_metric_rows', 'value': int(len(seed_metrics_df))},
        {'artifact': 'missing_manifest_rows', 'value': int(len(missing_df))},
        {'artifact': 'seed_grouped_pfi_rows', 'value': int(len(seed_grouped_pfi_df))},
        {'artifact': 'seed_grouped_pfi_fine_rows', 'value': int(len(seed_grouped_pfi_fine_df))},
        {'artifact': 'seed_grouped_shap_rows', 'value': int(len(seed_grouped_shap_df))},
        {'artifact': 'seed_grouped_shap_fine_rows', 'value': int(len(seed_grouped_shap_fine_df))},
        {'artifact': 'seed_agreement_rows', 'value': int(len(seed_agreement_df))},
        {'artifact': 'seed_agreement_fine_rows', 'value': int(len(seed_agreement_fine_df))},
        {'artifact': 'group_share_rows', 'value': int(len(group_share_summary_df))},
        {'artifact': 'group_share_fine_rows', 'value': int(len(group_share_summary_fine_df))},
    ]
)

if missing_df.empty:
    display(Markdown(f'Full manifest coverage available: **{completed_manifest_rows}/{len(full_manifest_df)}** slots ({manifest_coverage:.1%}).'))
else:
    display(Markdown(f'### Partial matrix coverage\nCompleted metric rows currently cover **{completed_manifest_rows}/{len(full_manifest_df)}** manifest slots ({manifest_coverage:.1%}).'))
    display(missing_df.head(12))

display(integrity_df)
display(Markdown('### Broad thesis rollup overview (first rows)'))
display(group_share_overview_df.head(24))
if not group_share_overview_fine_df.empty:
    display(Markdown('### Fine driver decomposition overview (first rows)'))
    display(group_share_overview_fine_df.head(36))
else:
    display(Markdown('### Fine driver decomposition overview\nFine grouped artifacts are not available yet in the current reload scope.'))
if driver_plot_rows:
    display(Markdown('### Saved primary driver-class plots'))
    display(pd.DataFrame(driver_plot_rows))
if fine_driver_plot_rows:
    display(Markdown('### Saved fine driver-decomposition plots'))
    display(pd.DataFrame(fine_driver_plot_rows))


Full manifest coverage available: **4860/4860** slots (100.0%).

,artifact,value
0,manifest_rows,4860
1,seed_metric_rows,4860
2,missing_manifest_rows,0
3,seed_grouped_pfi_rows,24624
4,seed_grouped_pfi_fine_rows,31752
5,seed_grouped_shap_rows,24624
6,seed_grouped_shap_fine_rows,31752
7,seed_agreement_rows,23004
8,seed_agreement_fine_rows,31752
9,group_share_rows,16416


### Broad thesis rollup overview (first rows)

,model_family,regime,weather_mode,mode,horizon_h,target_kind,method,feature_group,mean_share,mean_value,share_basis_mean,n_buildings,n_rows
0,xgboost,per_building,FW0,M0,2,cum,pfi,recent_demand_history,0.893809,140.622259,140.622259,6,6
1,xgboost,per_building,FW0,M0,2,cum,pfi,current_weather,0.080761,12.354849,12.354849,6,6
2,xgboost,per_building,FW0,M0,2,cum,pfi,calendar,0.025430,4.139482,4.139482,6,6
3,xgboost,per_building,FW0,M0,2,cum,shap,recent_demand_history,0.817843,85.012331,85.012331,6,6
4,xgboost,per_building,FW0,M0,2,cum,shap,current_weather,0.121582,11.800121,11.800121,6,6
5,xgboost,per_building,FW0,M0,2,cum,shap,calendar,0.060574,5.693191,5.693191,6,6
6,xgboost,per_building,FW0,M0,4,cum,pfi,recent_demand_history,0.741171,232.824793,232.824793,6,6
7,xgboost,per_building,FW0,M0,4,cum,pfi,current_weather,0.191119,56.932813,56.932813,6,6
8,xgboost,per_building,FW0,M0,4,cum,pfi,calendar,0.067710,22.896249,22.896249,6,6
9,xgboost,per_building,FW0,M0,4,cum,shap,recent_demand_history,0.692132,154.028628,154.028628,6,6


### Fine driver decomposition overview (first rows)

,model_family,regime,weather_mode,mode,horizon_h,target_kind,method,feature_group,mean_share,mean_value,share_basis_mean,n_buildings,n_rows
0,xgboost,per_building,FW0,M0,2,cum,pfi,demand_signal,0.893809,140.622259,140.622259,6,6
1,xgboost,per_building,FW0,M0,2,cum,pfi,current_weather,0.080761,12.354849,12.354849,6,6
2,xgboost,per_building,FW0,M0,2,cum,pfi,calendar,0.025430,4.139482,4.139482,6,6
3,xgboost,per_building,FW0,M0,2,cum,shap,demand_signal,0.817843,85.012331,85.012331,6,6
4,xgboost,per_building,FW0,M0,2,cum,shap,current_weather,0.121582,11.800121,11.800121,6,6
5,xgboost,per_building,FW0,M0,2,cum,shap,calendar,0.060574,5.693191,5.693191,6,6
6,xgboost,per_building,FW0,M0,4,cum,pfi,demand_signal,0.741171,232.824793,232.824793,6,6
7,xgboost,per_building,FW0,M0,4,cum,pfi,current_weather,0.191119,56.932813,56.932813,6,6
8,xgboost,per_building,FW0,M0,4,cum,pfi,calendar,0.067710,22.896249,22.896249,6,6
9,xgboost,per_building,FW0,M0,4,cum,shap,demand_signal,0.692132,154.028628,154.028628,6,6


### Saved primary driver-class plots

,taxonomy,method,path
0,broad,pfi,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
1,broad,shap,/Users/mihkeluutar/Documents/TalTech/Thesis/th...


### Saved fine driver-decomposition plots

,taxonomy,method,path
0,fine,pfi,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
1,fine,shap,/Users/mihkeluutar/Documents/TalTech/Thesis/th...



## Primary Thesis Output 2: What Changes When Inertia Features Are Added?

This section follows the mode-transition logic of the thesis grammar. Read the comparisons as `M1 - M0` for historical weather memory, `M2 - M0` for system dynamics, `M3 - M2` for additional historical weather memory once system dynamics are present, and `M4 - M3` for static building context. `FW0` remains the cleanest transition-reading slice because future weather is absent there.

### Static Context: Profile vs EHR Morphology

Once `M3` is available, the defended static comparison is `M4 - M3`, not `M4 - M0`. The fine taxonomy therefore splits the static branch into a regular profile-and-inventory block and an EHR-morphology block so that the notebook can show what kind of static signal is actually entering the pooled model.

### Why Per-Building Static Effects Are Limited

Within one building, the static descriptors are nearly constant across time. That means per-building static importance is structurally limited and should be treated as supporting context only. The pooled regime is the main thesis lens for static and EHR interpretation.


In [10]:

if primary_transition_df.empty and supporting_transition_df.empty:
    display(Markdown('No mode-transition summary rows are available yet.'))
else:
    if not primary_transition_df.empty:
        display(Markdown('### Primary thesis transitions'))
        display(primary_transition_df)
    if not supporting_transition_df.empty:
        display(Markdown('### Supporting transition context'))
        display(supporting_transition_df)

xgboost_story_modes = tuple(mode for mode in REQUESTED_MODES if mode in set(group_share_overview_df.loc[group_share_overview_df['model_family'] == 'xgboost', 'mode'].dropna().astype(str)))
inertia_feature_groups = tuple(
    feature_group
    for feature_group in ('historical_weather_memory', 'system_dynamic_proxies', 'static_building_context')
    if feature_group in set(group_share_overview_df.get('feature_group', pd.Series(dtype=object)).dropna().astype(str))
)
fine_transition_feature_groups = tuple(
    feature_group
    for feature_group in (
        'weather_memory',
        'space_heating_dynamics',
        'ventilation_dynamics',
        'static_profile_and_inventory',
        'static_ehr_morphology',
    )
    if feature_group in set(group_share_overview_fine_df.get('feature_group', pd.Series(dtype=object)).dropna().astype(str))
)
transition_plot_rows = []
fine_transition_plot_rows = []
if not group_share_overview_df.empty and xgboost_story_modes and 'FW0' in set(group_share_overview_df['weather_mode'].dropna().astype(str)):
    for method in ('pfi', 'shap'):
        save_path = artifact_paths['plots'] / f'thesis_mode_transition_fw0_{method}.png'
        fig = xns.plot_mode_feature_share_lines(
            group_share_overview_df,
            method=method,
            model_family='xgboost',
            regime='per_building',
            weather_mode='FW0',
            modes=xgboost_story_modes,
            feature_groups=inertia_feature_groups,
            taxonomy='broad',
            title=f'FW0 mode-share decomposition | XGBoost | per_building | {method.upper()}',
            save_path=save_path,
        )
        plt.close(fig)
        register_plot(
            save_path,
            plot_type=f'{method}_mode_transition_fw0',
            model_family='xgboost',
            regime='per_building',
            building='ALL',
            mode='ALL_CURRENT',
            weather_mode='FW0',
            horizon_h=pd.NA,
            thesis_role='primary',
            analysis_theme='inertia',
        )
        transition_plot_rows.append({'taxonomy': 'broad', 'method': method, 'path': str(save_path)})

if not group_share_overview_fine_df.empty and xgboost_story_modes and 'FW0' in set(group_share_overview_fine_df['weather_mode'].dropna().astype(str)):
    for method in ('pfi', 'shap'):
        save_path = artifact_paths['plots'] / f'thesis_mode_transition_fine_fw0_{method}.png'
        fig = xns.plot_mode_feature_share_lines(
            group_share_overview_fine_df,
            method=method,
            model_family='xgboost',
            regime='per_building',
            weather_mode='FW0',
            modes=xgboost_story_modes,
            feature_groups=fine_transition_feature_groups,
            taxonomy='fine',
            title=f'FW0 fine transition decomposition | XGBoost | per_building | {method.upper()}',
            save_path=save_path,
        )
        plt.close(fig)
        register_plot(
            save_path,
            plot_type=f'{method}_mode_transition_fine_fw0',
            model_family='xgboost',
            regime='per_building',
            building='ALL',
            mode='ALL_CURRENT',
            weather_mode='FW0',
            horizon_h=pd.NA,
            thesis_role='primary',
            analysis_theme='inertia_fine',
        )
        fine_transition_plot_rows.append({'taxonomy': 'fine', 'method': method, 'path': str(save_path)})

fine_transition_summary_df = group_share_overview_fine_df.loc[
    (group_share_overview_fine_df['model_family'] == 'xgboost')
    & (group_share_overview_fine_df['regime'] == 'per_building')
    & (group_share_overview_fine_df['weather_mode'] == 'FW0')
    & (group_share_overview_fine_df['feature_group'].isin(fine_transition_feature_groups))
].copy()
if not fine_transition_summary_df.empty:
    display(Markdown('### Fine-group transition summary in `FW0`'))
    display(
        fine_transition_summary_df.sort_values(['method', 'mode', 'horizon_h', 'feature_group'])
        .head(48)
        .reset_index(drop=True)
    )

static_fine_groups = tuple(
    feature_group
    for feature_group in ('static_profile_and_inventory', 'static_ehr_morphology')
    if feature_group in set(group_share_overview_fine_df.get('feature_group', pd.Series(dtype=object)).dropna().astype(str))
)
pooled_static_summary_df = group_share_overview_fine_df.loc[
    (group_share_overview_fine_df['model_family'] == 'xgboost')
    & (group_share_overview_fine_df['regime'] == 'pooled_same_buildings')
    & (group_share_overview_fine_df['mode'] == 'M4')
    & (group_share_overview_fine_df['feature_group'].isin(static_fine_groups))
].copy()
per_building_static_summary_df = group_share_overview_fine_df.loc[
    (group_share_overview_fine_df['model_family'] == 'xgboost')
    & (group_share_overview_fine_df['regime'] == 'per_building')
    & (group_share_overview_fine_df['mode'] == 'M4')
    & (group_share_overview_fine_df['feature_group'].isin(static_fine_groups))
].copy()
static_plot_rows = []
if not pooled_static_summary_df.empty and static_fine_groups:
    pooled_weather_modes = pooled_static_summary_df['weather_mode'].dropna().astype(str).tolist()
    static_focus_weather_mode = 'FW0' if 'FW0' in pooled_weather_modes else sorted(set(pooled_weather_modes))[0]
    for method in ('pfi', 'shap'):
        save_path = artifact_paths['plots'] / f'thesis_static_context_pooled_{method}.png'
        fig = xns.plot_feature_share_horizon_lines(
            group_share_overview_fine_df,
            method=method,
            model_family='xgboost',
            regime='pooled_same_buildings',
            weather_mode=static_focus_weather_mode,
            modes=('M4',),
            feature_groups=static_fine_groups,
            taxonomy='fine',
            title=f'Static context split | XGBoost | pooled_same_buildings | {static_focus_weather_mode} | {method.upper()}',
            save_path=save_path,
        )
        plt.close(fig)
        register_plot(
            save_path,
            plot_type=f'{method}_static_context_pooled',
            model_family='xgboost',
            regime='pooled_same_buildings',
            building='ALL',
            mode='M4',
            weather_mode=static_focus_weather_mode,
            horizon_h=pd.NA,
            thesis_role='primary',
            analysis_theme='static_context',
        )
        static_plot_rows.append({'method': method, 'weather_mode': static_focus_weather_mode, 'path': str(save_path)})

if transition_plot_rows:
    display(Markdown('### Saved FW0 mode-transition plots'))
    display(pd.DataFrame(transition_plot_rows))
if fine_transition_plot_rows:
    display(Markdown('### Saved fine FW0 mode-transition plots'))
    display(pd.DataFrame(fine_transition_plot_rows))

display(Markdown('### Static Context: Profile vs EHR Morphology'))
display(Markdown(
    'The defended static-context comparison in this notebook is `M4 - M3`. The pooled regime is the main lens because only pooled training can express cross-building static variation over time.'
))
if not pooled_static_summary_df.empty:
    display(
        pooled_static_summary_df.sort_values(['weather_mode', 'method', 'horizon_h', 'feature_group'])
        .head(48)
        .reset_index(drop=True)
    )
else:
    display(Markdown('No pooled static-context fine summary rows are available yet.'))
if static_plot_rows:
    display(Markdown('### Saved pooled static-context plots'))
    display(pd.DataFrame(static_plot_rows))

display(Markdown('### Why Per-Building Static Effects Are Limited'))
display(Markdown(
    'Within a single building, the static profile and EHR descriptors are almost constant across rows. Per-building static importance is therefore shown only as supporting context and should not carry the main static interpretation claim.'
))
if not per_building_static_summary_df.empty:
    display(
        per_building_static_summary_df.sort_values(['weather_mode', 'method', 'horizon_h', 'feature_group'])
        .head(24)
        .reset_index(drop=True)
    )
else:
    display(Markdown('No per-building static fine summary rows are available in the current reload scope.'))


### Primary thesis transitions

,model_family,regime,weather_mode,horizon_h,comparison,n_buildings,mean_delta_wape_pct_pp,median_delta_wape_pct_pp,buildings_improved_wape
0,xgboost,per_building,FW0,2,M1 - M0,6,0.114902,0.118260,0
1,xgboost,per_building,FW0,2,M2 - M0,6,0.035440,0.013982,3
2,xgboost,per_building,FW0,2,M3 - M2,6,0.033450,0.012633,3
3,xgboost,per_building,FW0,2,M4 - M3,6,0.122352,0.138755,0
4,xgboost,per_building,FW0,4,M1 - M0,6,0.136888,0.121554,1
...,...,...,...,...,...,...,...,...,...
211,xgboost,pooled_same_buildings,FW2,24,M4 - M3,6,-3.726785,-2.920300,4
212,xgboost,pooled_same_buildings,FW2,36,M1 - M0,6,-0.074851,-0.026709,3
213,xgboost,pooled_same_buildings,FW2,36,M2 - M0,6,-2.808573,-1.520176,4
214,xgboost,pooled_same_buildings,FW2,36,M3 - M2,6,-0.251426,-0.321690,4


### Supporting transition context

,model_family,regime,weather_mode,horizon_h,comparison,n_buildings,mean_delta_wape_pct_pp,median_delta_wape_pct_pp,buildings_improved_wape
0,xgboost,per_building,FW0,2,M3 - M1,6,-0.046013,-0.054062,3
1,xgboost,per_building,FW0,2,M4 - M0,6,0.191241,0.223497,2
2,xgboost,per_building,FW0,4,M3 - M1,6,-0.023330,-0.057043,4
3,xgboost,per_building,FW0,4,M4 - M0,6,0.219512,0.258138,1
4,xgboost,per_building,FW0,6,M3 - M1,6,0.014525,0.000653,3
...,...,...,...,...,...,...,...,...,...
103,xgboost,pooled_same_buildings,FW2,20,M4 - M0,6,-6.778491,-2.018637,6
104,xgboost,pooled_same_buildings,FW2,24,M3 - M1,6,-2.858008,-1.540256,4
105,xgboost,pooled_same_buildings,FW2,24,M4 - M0,6,-6.441172,-1.629569,6
106,xgboost,pooled_same_buildings,FW2,36,M3 - M1,6,-2.985148,-1.771282,4


### Fine-group transition summary in `FW0`

...

### Saved pooled static-context plots

,method,weather_mode,path
0,pfi,FW0,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
1,shap,FW0,/Users/mihkeluutar/Documents/TalTech/Thesis/th...


### Why Per-Building Static Effects Are Limited

Within a single building, the static profile and EHR descriptors are almost constant across rows. Per-building static importance is therefore shown only as supporting context and should not carry the main static interpretation claim.

,model_family,regime,weather_mode,mode,horizon_h,target_kind,method,feature_group,mean_share,mean_value,share_basis_mean,n_buildings,n_rows
0,xgboost,per_building,FW0,M4,2,cum,pfi,static_ehr_morphology,0.0,0.0,0.0,6,6
1,xgboost,per_building,FW0,M4,2,cum,pfi,static_profile_and_inventory,0.0,0.0,0.0,6,6
2,xgboost,per_building,FW0,M4,4,cum,pfi,static_ehr_morphology,0.0,0.0,0.0,6,6
3,xgboost,per_building,FW0,M4,4,cum,pfi,static_profile_and_inventory,0.0,0.0,0.0,6,6
4,xgboost,per_building,FW0,M4,6,cum,pfi,static_ehr_morphology,0.0,0.0,0.0,6,6
5,xgboost,per_building,FW0,M4,6,cum,pfi,static_profile_and_inventory,0.0,0.0,0.0,6,6
6,xgboost,per_building,FW0,M4,8,cum,pfi,static_ehr_morphology,0.0,0.0,0.0,6,6
7,xgboost,per_building,FW0,M4,8,cum,pfi,static_profile_and_inventory,0.0,0.0,0.0,6,6
8,xgboost,per_building,FW0,M4,12,cum,pfi,static_ehr_morphology,0.0,0.0,0.0,6,6
9,xgboost,per_building,FW0,M4,12,cum,pfi,static_profile_and_inventory,0.0,0.0,0.0,6,6



## Primary Thesis Output 3: How Does The Story Change In FW0 vs FW2 vs FW1?

The notebook reads the weather regimes asymmetrically on purpose. `FW0` is the clean inertia-identification slice, `FW2` is the realistic forecast-like slice, and `FW1` is the oracle upper bound. The plots below keep those roles explicit instead of treating all three weather settings as interchangeable.

### Past Weather vs Future Weather

The broad weather-role plots stay in place for the headline story, but the fine taxonomy is used underneath them to separate current weather, historical weather memory, future-weather summaries, and future-weather paths. This is the main notebook-side answer to the question of how past weather and future weather affect the retained models differently.


In [11]:

plots_dir = Path(artifact_paths['plots'])
plots_dir.mkdir(parents=True, exist_ok=True)

coverage_summary_df = xns.build_manifest_coverage_summary(manifest_df, seed_metrics_df)
completed_slots = int(coverage_summary_df['completed_slots'].sum()) if not coverage_summary_df.empty else 0
total_slots = int(coverage_summary_df['total_slots'].sum()) if not coverage_summary_df.empty else 0
coverage_fraction = (completed_slots / total_slots) if total_slots else float('nan')
plot_file_count = len(list(plots_dir.glob('*.png')))

section_status_df = pd.DataFrame(
    [
        {
            'section': 'Saved artifact reload',
            'status': 'loaded' if not seed_metrics_df.empty else 'missing',
            'detail': f'{len(seed_metrics_df)} seed metric rows are available from disk.',
        },
        {
            'section': 'Current thesis mode grammar',
            'status': 'complete' if not mode_completion_df.empty and bool((mode_completion_df['status'] == 'complete').all()) else 'partial',
            'detail': ', '.join(f"{row.mode}:{row.status}" for row in mode_completion_df.itertuples(index=False)) if not mode_completion_df.empty else '(no rows)',
        },
        {
            'section': 'Manifest coverage',
            'status': 'complete' if not coverage_summary_df.empty and bool((coverage_summary_df['coverage_pct'] == 1.0).all()) else 'partial',
            'detail': f'{completed_slots}/{total_slots} completed slots across the current manifest coverage summary.',
        },
        {
            'section': 'Primary weather-role plots',
            'status': 'ready' if not group_share_overview_df.empty else 'missing',
            'detail': f'PNG files currently under plots/: {plot_file_count}.',
        },
    ]
)
display(Markdown('### Completion Snapshot'))
display(section_status_df)

if not coverage_summary_df.empty:
    coverage_heatmap_path = plots_dir / 'thesis_coverage_heatmap.png'
    fig = xns.plot_manifest_coverage_heatmap(
        coverage_summary_df,
        title='Current notebook completion by regime, weather mode, and horizon',
        save_path=coverage_heatmap_path,
    )
    plt.close(fig)
    register_plot(
        coverage_heatmap_path,
        plot_type='coverage_heatmap',
        model_family='ALL_CURRENT',
        regime='ALL',
        building='ALL',
        mode='ALL_CURRENT',
        weather_mode='ALL_CURRENT',
        horizon_h=pd.NA,
        thesis_role='primary',
        analysis_theme='scope',
    )
    display(Markdown('### Weather-role completion grid'))
    display(mode_weather_completion_df)

weather_plot_rows = []
fine_weather_plot_rows = []
xgboost_story_modes = tuple(mode for mode in REQUESTED_MODES if mode in set(group_share_overview_df.loc[group_share_overview_df['model_family'] == 'xgboost', 'mode'].dropna().astype(str)))
available_feature_groups = tuple(
    feature_group
    for feature_group in xns.THESIS_BROAD_FEATURE_GROUP_ORDER
    if feature_group in set(group_share_overview_df.get('feature_group', pd.Series(dtype=object)).dropna().astype(str))
)
fine_weather_feature_groups = tuple(
    feature_group
    for feature_group in (
        'current_weather',
        'weather_memory',
        'future_weather_summaries',
        'future_weather_paths',
    )
    if feature_group in set(group_share_overview_fine_df.get('feature_group', pd.Series(dtype=object)).dropna().astype(str))
)
for weather_mode in REQUESTED_WEATHER_MODES:
    weather_slice = group_share_overview_df.loc[
        (group_share_overview_df['model_family'] == 'xgboost')
        & (group_share_overview_df['regime'] == 'per_building')
        & (group_share_overview_df['weather_mode'] == weather_mode)
    ].copy()
    if weather_slice.empty or not xgboost_story_modes:
        continue
    for method in ('pfi', 'shap'):
        save_path = plots_dir / f'thesis_weather_role_{weather_mode}_{method}.png'
        fig = xns.plot_feature_share_horizon_lines(
            group_share_overview_df,
            method=method,
            model_family='xgboost',
            regime='per_building',
            weather_mode=weather_mode,
            modes=xgboost_story_modes,
            feature_groups=available_feature_groups,
            taxonomy='broad',
            title=f'Weather-role comparison | XGBoost | per_building | {weather_mode} | {method.upper()}',
            save_path=save_path,
        )
        plt.close(fig)
        register_plot(
            save_path,
            plot_type=f'{method}_weather_role_{weather_mode.lower()}',
            model_family='xgboost',
            regime='per_building',
            building='ALL',
            mode='ALL_CURRENT',
            weather_mode=weather_mode,
            horizon_h=pd.NA,
            thesis_role='primary',
            analysis_theme='weather_information',
        )
        weather_plot_rows.append({'taxonomy': 'broad', 'weather_mode': weather_mode, 'method': method, 'path': str(save_path)})

for weather_mode in REQUESTED_WEATHER_MODES:
    fine_weather_slice = group_share_overview_fine_df.loc[
        (group_share_overview_fine_df['model_family'] == 'xgboost')
        & (group_share_overview_fine_df['regime'] == 'per_building')
        & (group_share_overview_fine_df['weather_mode'] == weather_mode)
    ].copy()
    if fine_weather_slice.empty or not xgboost_story_modes or not fine_weather_feature_groups:
        continue
    for method in ('pfi', 'shap'):
        save_path = plots_dir / f'thesis_weather_role_fine_{weather_mode}_{method}.png'
        fig = xns.plot_feature_share_horizon_lines(
            group_share_overview_fine_df,
            method=method,
            model_family='xgboost',
            regime='per_building',
            weather_mode=weather_mode,
            modes=xgboost_story_modes,
            feature_groups=fine_weather_feature_groups,
            taxonomy='fine',
            title=f'Past vs future weather split | XGBoost | per_building | {weather_mode} | {method.upper()}',
            save_path=save_path,
        )
        plt.close(fig)
        register_plot(
            save_path,
            plot_type=f'{method}_weather_role_fine_{weather_mode.lower()}',
            model_family='xgboost',
            regime='per_building',
            building='ALL',
            mode='ALL_CURRENT',
            weather_mode=weather_mode,
            horizon_h=pd.NA,
            thesis_role='primary',
            analysis_theme='weather_information_fine',
        )
        fine_weather_plot_rows.append({'taxonomy': 'fine', 'weather_mode': weather_mode, 'method': method, 'path': str(save_path)})

display(Markdown('### Weather-role table'))
display(weather_role_df)
if weather_plot_rows:
    display(Markdown('### Saved weather-role plots'))
    display(pd.DataFrame(weather_plot_rows))
else:
    display(Markdown('No weather-role plots could be prepared from the current grouped-share summary.'))

display(Markdown('### Past Weather vs Future Weather'))
if not group_share_overview_fine_df.empty and fine_weather_feature_groups:
    fine_weather_summary_df = group_share_overview_fine_df.loc[
        (group_share_overview_fine_df['model_family'] == 'xgboost')
        & (group_share_overview_fine_df['regime'] == 'per_building')
        & (group_share_overview_fine_df['feature_group'].isin(fine_weather_feature_groups))
    ].copy()
    display(
        fine_weather_summary_df.sort_values(['weather_mode', 'method', 'mode', 'horizon_h', 'feature_group'])
        .head(64)
        .reset_index(drop=True)
    )
    if fine_weather_plot_rows:
        display(Markdown('### Saved fine weather-role plots'))
        display(pd.DataFrame(fine_weather_plot_rows))
else:
    display(Markdown('Fine weather-group plots are not available yet in the current reload scope.'))


### Completion Snapshot

,section,status,detail
0,Saved artifact reload,loaded,4860 seed metric rows are available from disk.
1,Current thesis mode grammar,complete,"M0:complete, M1:complete, M2:complete, M3:comp..."
2,Manifest coverage,complete,4860/4860 completed slots across the current m...
3,Primary weather-role plots,ready,PNG files currently under plots/: 335.


### Weather-role completion grid

,regime,weather_mode,mode,status,completed_slots,total_slots,coverage_pct
0,per_building,FW0,M0,complete,162,162,1.0
1,per_building,FW0,M1,complete,162,162,1.0
2,per_building,FW0,M2,complete,162,162,1.0
3,per_building,FW0,M3,complete,162,162,1.0
4,per_building,FW0,M4,complete,162,162,1.0
5,per_building,FW2,M0,complete,162,162,1.0
6,per_building,FW2,M1,complete,162,162,1.0
7,per_building,FW2,M2,complete,162,162,1.0
8,per_building,FW2,M3,complete,162,162,1.0
9,per_building,FW2,M4,complete,162,162,1.0


### Weather-role table

...

,taxonomy,weather_mode,method,path
0,broad,FW0,pfi,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
1,broad,FW0,shap,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
2,broad,FW2,pfi,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
3,broad,FW2,shap,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
4,broad,FW1,pfi,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
5,broad,FW1,shap,/Users/mihkeluutar/Documents/TalTech/Thesis/th...


### Past Weather vs Future Weather

,model_family,regime,weather_mode,mode,horizon_h,target_kind,method,feature_group,mean_share,mean_value,share_basis_mean,n_buildings,n_rows
0,xgboost,per_building,FW0,M0,2,cum,pfi,current_weather,0.080761,12.354849,12.354849,6,6
1,xgboost,per_building,FW0,M0,4,cum,pfi,current_weather,0.191119,56.932813,56.932813,6,6
2,xgboost,per_building,FW0,M0,6,cum,pfi,current_weather,0.255679,114.875302,114.875302,6,6
3,xgboost,per_building,FW0,M0,8,cum,pfi,current_weather,0.301043,178.525465,178.525465,6,6
4,xgboost,per_building,FW0,M0,12,cum,pfi,current_weather,0.352543,299.178495,299.178495,6,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,xgboost,per_building,FW0,M4,6,cum,pfi,weather_memory,0.055011,21.805669,22.232080,6,6
60,xgboost,per_building,FW0,M4,8,cum,pfi,current_weather,0.254519,136.139960,136.139960,6,6
61,xgboost,per_building,FW0,M4,8,cum,pfi,weather_memory,0.083547,42.558632,43.127515,6,6
62,xgboost,per_building,FW0,M4,12,cum,pfi,current_weather,0.280050,220.125709,220.125709,6,6


### Saved fine weather-role plots

,taxonomy,weather_mode,method,path
0,fine,FW0,pfi,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
1,fine,FW0,shap,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
2,fine,FW2,pfi,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
3,fine,FW2,shap,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
4,fine,FW1,pfi,/Users/mihkeluutar/Documents/TalTech/Thesis/th...
5,fine,FW1,shap,/Users/mihkeluutar/Documents/TalTech/Thesis/th...



## Primary Thesis Output 4: Are The Explanations Stable And Consistent?

The thesis reads grouped PFI and grouped SHAP together. Agreement and repeated-seed stability do not prove causal truth, but they do show whether the explanation structure is coherent enough to support bounded interpretation.

### Fine-Group Stability Check

The broad stability table remains the headline checkpoint because it matches the thesis-facing driver vocabulary. The fine stability table is a diagnostic check: it shows whether the more granular decomposition stays coherent enough to be worth reading at all.


In [12]:

if stability_overview_df.empty:
    display(Markdown('No aggregated agreement/stability rows are available yet.'))
else:
    xgboost_stability_df = stability_overview_df.loc[
        stability_overview_df['model_family'] == 'xgboost'
    ].copy()
    display(Markdown('### Aggregated XGBoost agreement and stability overview (broad rollup)'))
    display(xgboost_stability_df)

if stability_overview_fine_df.empty:
    display(Markdown('### Fine-group stability check\nNo fine-group agreement/stability rows are available yet.'))
else:
    xgboost_stability_fine_df = stability_overview_fine_df.loc[
        stability_overview_fine_df['model_family'] == 'xgboost'
    ].copy()
    display(Markdown('### Aggregated XGBoost agreement and stability overview (fine taxonomy)'))
    display(xgboost_stability_fine_df)


### Aggregated XGBoost agreement and stability overview (broad rollup)

,model_family,regime,weather_mode,mode,horizon_h,target_kind,n_buildings,consensus_top_pfi_group,consensus_top_shap_group,mean_rank_gap,max_rank_gap,pairwise_spearman_mean_pfi,pairwise_spearman_mean_shap,topk_overlap_mean_pfi,topk_overlap_mean_shap
0,xgboost,per_building,FW0,M0,2,cum,6,recent_demand_history,recent_demand_history,0.074074,0.666667,1.000000,0.944444,1.000000,1.0
1,xgboost,per_building,FW0,M0,4,cum,6,recent_demand_history,recent_demand_history,0.074074,0.666667,1.000000,0.944444,1.000000,1.0
2,xgboost,per_building,FW0,M0,6,cum,6,recent_demand_history,recent_demand_history,0.000000,0.000000,1.000000,1.000000,1.000000,1.0
3,xgboost,per_building,FW0,M0,8,cum,6,recent_demand_history,recent_demand_history,0.000000,0.000000,1.000000,1.000000,1.000000,1.0
4,xgboost,per_building,FW0,M0,12,cum,6,recent_demand_history,recent_demand_history,0.111111,1.000000,1.000000,1.000000,1.000000,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,xgboost,pooled_same_buildings,FW1,M4,12,cum,6,recent_demand_history,recent_demand_history,1.603175,5.000000,0.988095,0.992063,0.962963,1.0
266,xgboost,pooled_same_buildings,FW1,M4,16,cum,6,recent_demand_history,recent_demand_history,1.674603,5.000000,0.986111,1.000000,1.000000,1.0
267,xgboost,pooled_same_buildings,FW1,M4,20,cum,6,recent_demand_history,recent_demand_history,1.825397,5.000000,0.988095,0.988095,0.962963,1.0
268,xgboost,pooled_same_buildings,FW1,M4,24,cum,6,recent_demand_history,recent_demand_history,1.746032,5.000000,0.980159,0.996032,0.962963,1.0


### Aggregated XGBoost agreement and stability overview (fine taxonomy)

,model_family,regime,weather_mode,mode,horizon_h,target_kind,n_buildings,consensus_top_pfi_group,consensus_top_shap_group,mean_rank_gap,max_rank_gap,pairwise_spearman_mean_pfi,pairwise_spearman_mean_shap,topk_overlap_mean_pfi,topk_overlap_mean_shap
0,xgboost,per_building,FW0,M0,2,cum,6,demand_signal,demand_signal,0.074074,0.666667,1.000000,0.944444,1.000000,1.000000
1,xgboost,per_building,FW0,M0,4,cum,6,demand_signal,demand_signal,0.074074,0.666667,1.000000,0.944444,1.000000,1.000000
2,xgboost,per_building,FW0,M0,6,cum,6,demand_signal,demand_signal,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
3,xgboost,per_building,FW0,M0,8,cum,6,demand_signal,demand_signal,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
4,xgboost,per_building,FW0,M0,12,cum,6,demand_signal,demand_signal,0.111111,1.000000,1.000000,1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,xgboost,pooled_same_buildings,FW1,M4,12,cum,6,demand_signal,demand_signal,1.577778,7.000000,0.980352,0.980471,0.962963,0.925926
266,xgboost,pooled_same_buildings,FW1,M4,16,cum,6,demand_signal,demand_signal,1.622222,7.000000,0.988482,0.985185,0.962963,0.925926
267,xgboost,pooled_same_buildings,FW1,M4,20,cum,6,demand_signal,demand_signal,1.588889,7.000000,0.981707,0.979798,1.000000,0.777778
268,xgboost,pooled_same_buildings,FW1,M4,24,cum,6,demand_signal,demand_signal,1.627778,7.000000,0.980352,0.991246,1.000000,0.925926


## Supporting Diagnostics: Anchor Buildings


In [13]:
detail_cache = matrix_artifacts['detail_cache'] if matrix_artifacts else saved_outputs.get('detail_cache', {})
detail_cache = rebuild_detail_cache(
    detail_cache,
    {
        (model_family, 'per_building', building, mode, FOCUS_WEATHER_MODE, horizon_h, DETAIL_SEED)
        for model_family in MODEL_FAMILIES
        for building in ANCHOR_BUILDINGS
        for mode in ACTIVE_XAI_MODES
        for horizon_h in DETAIL_HORIZONS
    },
    label='anchor SHAP plots',
)
anchor_plot_skips = []

for model_family in MODEL_FAMILIES:
    for building in ANCHOR_BUILDINGS:
        for mode in ACTIVE_XAI_MODES:
            for horizon_h in DETAIL_HORIZONS:
                detail_key = (model_family, 'per_building', building, mode, FOCUS_WEATHER_MODE, horizon_h, DETAIL_SEED)
                if detail_key not in detail_cache:
                    anchor_plot_skips.append({'model_family': model_family, 'building': building, 'mode': mode, 'horizon_h': horizon_h, 'reason': 'Detail cache entry is missing.'})
                    continue
                detail = detail_cache[detail_key]
                experiment = detail['experiment']
                shap_payload = detail['shap']
                shap_matrix, shap_feature_df = xns.prepare_shap_plot_payload(experiment, shap_payload)
                if shap_matrix.shape[1] != shap_feature_df.shape[1]:
                    anchor_plot_skips.append({'model_family': model_family, 'building': building, 'mode': mode, 'horizon_h': horizon_h, 'reason': 'SHAP matrix/feature shape mismatch.'})
                    continue

                violin_path = artifact_paths['plots'] / f'shap_violin_{model_family}_{building}_{mode}_h{horizon_h:02d}.png'
                fig = xns.plot_shap_violin_summary(
                    shap_matrix,
                    shap_feature_df,
                    max_display=12,
                    title=f'SHAP violin | {model_family} | {building} | {mode} | h={horizon_h}',
                    save_path=violin_path,
                )
                plt.close(fig)
                register_plot(violin_path, plot_type='shap_violin', model_family=model_family, regime='per_building', building=building, mode=mode, weather_mode=FOCUS_WEATHER_MODE, horizon_h=horizon_h, thesis_role='supporting', analysis_theme='supporting_case')

                beeswarm_path = artifact_paths['plots'] / f'shap_beeswarm_{model_family}_{building}_{mode}_h{horizon_h:02d}.png'
                fig = xns.plot_shap_beeswarm(
                    shap_matrix,
                    shap_feature_df,
                    max_display=12,
                    save_path=beeswarm_path,
                )
                plt.close(fig)
                register_plot(beeswarm_path, plot_type='shap_beeswarm', model_family=model_family, regime='per_building', building=building, mode=mode, weather_mode=FOCUS_WEATHER_MODE, horizon_h=horizon_h, thesis_role='supporting', analysis_theme='supporting_case')

                if model_family == 'xgboost':
                    top_features = (
                        pd.Series(np.abs(shap_matrix).mean(axis=0), index=shap_feature_df.columns)
                        .sort_values(ascending=False)
                        .head(3)
                        .index
                        .tolist()
                    )
                    for feature_name in top_features:
                        scatter_path = artifact_paths['plots'] / f'shap_scatter_{model_family}_{building}_{mode}_h{horizon_h:02d}_{feature_name}.png'
                        fig = xns.plot_shap_dependence_scatter(
                            shap_matrix,
                            shap_feature_df,
                            feature_name=feature_name,
                            title=f'SHAP dependence | {building} | {mode} | h={horizon_h} | {feature_name}',
                            save_path=scatter_path,
                        )
                        plt.close(fig)
                        register_plot(scatter_path, plot_type='shap_scatter', model_family=model_family, regime='per_building', building=building, mode=mode, weather_mode=FOCUS_WEATHER_MODE, horizon_h=horizon_h, feature_name=feature_name, thesis_role='supporting', analysis_theme='supporting_case')

plot_manifest_df = pd.DataFrame(plot_manifest_rows)
if plot_manifest_df.empty:
    display(Markdown('No anchor SHAP plots were generated from the currently available detail cache.'))
else:
    display(plot_manifest_df.groupby(['plot_type'], as_index=False).size())
if anchor_plot_skips:
    display(pd.DataFrame(anchor_plot_skips))


/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,plot_type,size
0,coverage_heatmap,1
1,pfi_driver_classes,1
2,pfi_driver_classes_fine,1
3,pfi_mode_transition_fine_fw0,1
4,pfi_mode_transition_fw0,1
5,pfi_static_context_pooled,1
6,pfi_weather_role_fine_fw0,1
7,pfi_weather_role_fine_fw1,1
8,pfi_weather_role_fine_fw2,1
9,pfi_weather_role_fw0,1


,model_family,building,mode,horizon_h,reason
0,xgboost,U05,M0,8,Detail cache entry is missing.
1,xgboost,U05,M3,8,Detail cache entry is missing.
2,xgboost,U05,M3,24,Detail cache entry is missing.
3,xgboost,U05,M3,36,Detail cache entry is missing.
4,xgboost,U06,M0,8,Detail cache entry is missing.
5,xgboost,U06,M3,8,Detail cache entry is missing.
6,xgboost,U06,M3,24,Detail cache entry is missing.
7,xgboost,U06,M3,36,Detail cache entry is missing.


## Supporting Diagnostics: Local Cases


In [14]:
detail_cache = matrix_artifacts['detail_cache'] if matrix_artifacts else saved_outputs.get('detail_cache', {})
detail_cache = rebuild_detail_cache(
    detail_cache,
    {
        ('xgboost', 'per_building', building, 'M0', FOCUS_WEATHER_MODE, 8, DETAIL_SEED)
        for building in ANCHOR_BUILDINGS
    },
    label='local-case selection',
)
selected_case_skips = []

if 'xgboost' not in MODEL_FAMILIES:
    selected_cases_df = pd.DataFrame()
    display(Markdown('Local-case selection is skipped because `RUN_XGBOOST_FAMILY = False`.'))
else:
    selected_case_frames = []
    for building in ANCHOR_BUILDINGS:
        reference_key = ('xgboost', 'per_building', building, 'M0', FOCUS_WEATHER_MODE, 8, DETAIL_SEED)
        if reference_key not in detail_cache:
            selected_case_skips.append({'building': building, 'reason': 'Reference detail slice is missing.'})
            continue
        reference_experiment = detail_cache[reference_key]['experiment']
        case_df = xns.select_representative_cases(reference_experiment).copy()
        if case_df.empty:
            selected_case_skips.append({'building': building, 'reason': 'Representative-case selection returned no rows.'})
            continue
        case_df['building'] = building
        case_df['selection_model_family'] = 'xgboost'
        case_df['selection_mode'] = 'M0'
        case_df['selection_weather_mode'] = FOCUS_WEATHER_MODE
        case_df['selection_horizon_h'] = 8
        case_df['selection_seed'] = DETAIL_SEED
        selected_case_frames.append(case_df)

    if selected_case_frames:
        selected_cases_df = pd.concat(selected_case_frames, ignore_index=True)
        selected_cases_df.to_csv(artifact_paths['cases'], index=False)
        display(selected_cases_df)
    else:
        selected_cases_df = pd.DataFrame()
        display(Markdown('Local-case selection found no usable anchor detail slices yet.'))

if selected_case_skips:
    display(pd.DataFrame(selected_case_skips))


Local-case selection found no usable anchor detail slices yet.

,building,reason
0,U05,Reference detail slice is missing.
1,U06,Reference detail slice is missing.


In [15]:
detail_cache = matrix_artifacts['detail_cache'] if matrix_artifacts else saved_outputs.get('detail_cache', {})
detail_cache = rebuild_detail_cache(
    detail_cache,
    {
        (model_family, 'per_building', building, mode, FOCUS_WEATHER_MODE, horizon_h, DETAIL_SEED)
        for model_family in MODEL_FAMILIES
        for building in ANCHOR_BUILDINGS
        for mode in ACTIVE_XAI_MODES
        for horizon_h in LOCAL_HORIZONS
    },
    label='local-case figures',
)
local_figure_skips = []

if selected_cases_df.empty:
    local_figure_df = pd.DataFrame()
    display(Markdown('Local-case figures are skipped because no selected cases are available.'))
else:
    local_rows = []
    for model_family in MODEL_FAMILIES:
        for building in ANCHOR_BUILDINGS:
            building_cases = selected_cases_df.loc[selected_cases_df['building'] == building].copy()
            if building_cases.empty:
                local_figure_skips.append({'model_family': model_family, 'building': building, 'reason': 'No selected cases available for this building.'})
                continue
            for mode in ACTIVE_XAI_MODES:
                for horizon_h in LOCAL_HORIZONS:
                    detail_key = (model_family, 'per_building', building, mode, FOCUS_WEATHER_MODE, horizon_h, DETAIL_SEED)
                    if detail_key not in detail_cache:
                        local_figure_skips.append({'model_family': model_family, 'building': building, 'mode': mode, 'horizon_h': horizon_h, 'reason': 'Local-detail slice is missing.'})
                        continue
                    detail = detail_cache[detail_key]
                    experiment = detail['experiment']
                    shap_payload = detail['shap']
                    for case in building_cases.itertuples(index=False):
                        match = experiment.predictions_df.loc[
                            pd.to_datetime(experiment.predictions_df['datetime']) == pd.Timestamp(case.datetime),
                            'row_idx',
                        ]
                        if match.empty:
                            local_figure_skips.append({'model_family': model_family, 'building': building, 'mode': mode, 'horizon_h': horizon_h, 'case_type': case.case_type, 'reason': 'Case datetime is missing from predictions.'})
                            continue
                        row_idx = int(match.iloc[0])
                        save_path = artifact_paths['plots'] / (
                            f'local_case_{model_family}_{building}_{mode}_h{horizon_h:02d}_{case.case_type}.png'
                        )
                        fig = xns.plot_local_case_figure_any(
                            experiment,
                            row_idx=row_idx,
                            case_type=str(case.case_type),
                            shap_payload=shap_payload,
                            save_path=save_path,
                            context_hours=24,
                            top_k=8,
                            random_seed=DETAIL_SEED,
                        )
                        plt.close(fig)
                        register_plot(save_path, plot_type='local_case', model_family=model_family, regime='per_building', building=building, mode=mode, weather_mode=FOCUS_WEATHER_MODE, horizon_h=horizon_h, case_type=str(case.case_type), thesis_role='supporting', analysis_theme='supporting_case')
                        local_rows.append(
                            {
                                'model_family': model_family,
                                'regime': 'per_building',
                                'building': building,
                                'mode': mode,
                                'weather_mode': FOCUS_WEATHER_MODE,
                                'horizon_h': horizon_h,
                                'case_type': case.case_type,
                                'datetime': case.datetime,
                                'plot_path': str(save_path),
                            }
                        )

    if local_rows:
        local_figure_df = pd.DataFrame(local_rows).sort_values(['model_family', 'building', 'horizon_h', 'mode', 'case_type']).reset_index(drop=True)
        local_figure_df.to_csv(artifact_paths['local_figures'], index=False)
        plot_manifest_df = pd.DataFrame(plot_manifest_rows)
        display(local_figure_df.head(24))
    else:
        local_figure_df = pd.DataFrame()
        display(Markdown('Local-case figure generation found no runnable slices yet.'))

if local_figure_skips:
    display(pd.DataFrame(local_figure_skips))


Local-case figures are skipped because no selected cases are available.

## Supporting Diagnostics: Operational Scenarios


In [16]:
detail_cache = matrix_artifacts['detail_cache'] if matrix_artifacts else saved_outputs.get('detail_cache', {})
detail_cache = rebuild_detail_cache(
    detail_cache,
    {
        ('xgboost', 'per_building', building, mode, FOCUS_WEATHER_MODE, horizon_h, DETAIL_SEED)
        for building in ANCHOR_BUILDINGS
        for mode in ACTIVE_XAI_MODES
        for horizon_h in LOCAL_HORIZONS
    },
    label='operational scenarios',
)
scenario_skips = []

if 'xgboost' not in MODEL_FAMILIES or selected_cases_df.empty:
    scenario_results_df = pd.DataFrame()
    scenario_summary_df = pd.DataFrame()
    rq3_summary_df = pd.DataFrame()
    display(Markdown('Scenario probes are skipped because `RUN_XGBOOST_FAMILY = False` or no selected cases are available.'))
else:
    scenario_frames = []
    for building in ANCHOR_BUILDINGS:
        building_cases = selected_cases_df.loc[
            (selected_cases_df['building'] == building)
            & (selected_cases_df['case_type'].isin(SCENARIO_CASE_TYPES))
        ].copy()
        if building_cases.empty:
            scenario_skips.append({'building': building, 'reason': 'No scenario case types are available for this building.'})
            continue
        for mode in ACTIVE_XAI_MODES:
            for horizon_h in LOCAL_HORIZONS:
                detail_key = ('xgboost', 'per_building', building, mode, FOCUS_WEATHER_MODE, horizon_h, DETAIL_SEED)
                if detail_key not in detail_cache:
                    scenario_skips.append({'building': building, 'mode': mode, 'horizon_h': horizon_h, 'reason': 'Scenario-detail slice is missing.'})
                    continue
                detail = detail_cache[detail_key]
                experiment = detail['experiment']
                for case in building_cases.itertuples(index=False):
                    match = experiment.predictions_df.loc[
                        pd.to_datetime(experiment.predictions_df['datetime']) == pd.Timestamp(case.datetime),
                        'row_idx',
                    ]
                    if match.empty:
                        scenario_skips.append({'building': building, 'mode': mode, 'horizon_h': horizon_h, 'case_type': case.case_type, 'reason': 'Scenario datetime is missing from predictions.'})
                        continue
                    row_idx = int(match.iloc[0])
                    scenario_df = xns.run_operational_scenarios(
                        experiment,
                        row_idx=row_idx,
                        shift_hours=(1, 2),
                        include_weather_memory=True,
                    )
                    scenario_df['model_family'] = 'xgboost'
                    scenario_df['regime'] = 'per_building'
                    scenario_df['building'] = building
                    scenario_df['mode'] = mode
                    scenario_df['weather_mode'] = FOCUS_WEATHER_MODE
                    scenario_df['horizon_h'] = horizon_h
                    scenario_df['case_type'] = case.case_type
                    scenario_df['datetime'] = case.datetime
                    scenario_frames.append(scenario_df)

                    plot_path = artifact_paths['plots'] / (
                        f'scenario_xgboost_{building}_{mode}_h{horizon_h:02d}_{case.case_type}.png'
                    )
                    fig = xns.plot_scenario_deltas(
                        scenario_df,
                        title=f'Scenario delta | {building} | {mode} | h={horizon_h} | {case.case_type}',
                        save_path=plot_path,
                    )
                    plt.close(fig)
                    register_plot(plot_path, plot_type='scenario', model_family='xgboost', regime='per_building', building=building, mode=mode, weather_mode=FOCUS_WEATHER_MODE, horizon_h=horizon_h, case_type=str(case.case_type), thesis_role='supporting', analysis_theme='supporting_case')

    if scenario_frames:
        scenario_results_df = pd.concat(scenario_frames, ignore_index=True)
        scenario_summary_df = xns.summarize_scenario_directionality(scenario_results_df)
        rq3_summary_df = xns.build_rq3_operational_summary(scenario_results_df)

        scenario_results_df.to_csv(artifact_paths['scenarios'], index=False)
        scenario_summary_df.to_csv(artifact_paths['scenario_summary'], index=False)
        rq3_summary_df.to_csv(artifact_paths['rq3_summary'], index=False)

        display(scenario_summary_df.head(24))
        display(rq3_summary_df.head(24))
    else:
        scenario_results_df = pd.DataFrame()
        scenario_summary_df = pd.DataFrame()
        rq3_summary_df = pd.DataFrame()
        display(Markdown('Scenario probes found no runnable slices yet.'))

if scenario_skips:
    display(pd.DataFrame(scenario_skips))


Scenario probes are skipped because `RUN_XGBOOST_FAMILY = False` or no selected cases are available.


## Final Artifact Manifests

The final manifest now covers both the broad thesis rollup and the fine diagnostic taxonomy so that the notebook can be reread either as a compact thesis artifact or as a more granular grouped-XAI audit.


In [17]:

if plot_manifest_rows:
    plot_manifest_df = pd.DataFrame(plot_manifest_rows)
    for col in PLOT_MANIFEST_COLUMNS:
        if col not in plot_manifest_df.columns:
            plot_manifest_df[col] = pd.NA
    plot_manifest_df = plot_manifest_df.sort_values(
        [
            'thesis_role',
            'analysis_theme',
            'plot_type',
            'model_family',
            'regime',
            'building',
            'mode',
            'weather_mode',
            'horizon_h',
            'case_type',
            'feature_name',
        ]
    ).reset_index(drop=True)
    plot_manifest_df.to_csv(artifact_paths['plot_manifest'], index=False)
else:
    plot_manifest_df = pd.DataFrame()
    display(Markdown('No plots have been registered yet; the manifest stays empty until runnable slices are available.'))

final_counts_df = pd.DataFrame(
    [
        {'artifact': 'plots', 'count': int(len(plot_manifest_df))},
        {'artifact': 'selected_cases', 'count': int(len(selected_cases_df))},
        {'artifact': 'local_figures', 'count': int(len(local_figure_df))},
        {'artifact': 'scenario_rows', 'count': int(len(scenario_results_df))},
        {'artifact': 'scenario_summary_rows', 'count': int(len(scenario_summary_df))},
        {'artifact': 'grouped_pfi_rows_broad', 'count': int(len(grouped_pfi_df))},
        {'artifact': 'grouped_pfi_rows_fine', 'count': int(len(grouped_pfi_fine_df))},
        {'artifact': 'grouped_shap_rows_broad', 'count': int(len(grouped_shap_df))},
        {'artifact': 'grouped_shap_rows_fine', 'count': int(len(grouped_shap_fine_df))},
        {'artifact': 'rq1_rows', 'count': int(len(rq1_summary_df))},
        {'artifact': 'rq2_rows_broad', 'count': int(len(rq2_summary_df))},
        {'artifact': 'rq2_rows_fine', 'count': int(len(rq2_summary_fine_df))},
        {'artifact': 'rq3_rows', 'count': int(len(rq3_summary_df))},
    ]
)

display(final_counts_df)
if not plot_manifest_df.empty:
    display(Markdown('### Plot manifest by thesis role'))
    display(plot_manifest_df.groupby(['thesis_role', 'analysis_theme', 'plot_type'], as_index=False).size())
    display(plot_manifest_df.head(24))


,artifact,count
0,plots,133
1,selected_cases,0
2,local_figures,0
3,scenario_rows,0
4,scenario_summary_rows,0
5,grouped_pfi_rows_broad,8208
6,grouped_pfi_rows_fine,10584
7,grouped_shap_rows_broad,8208
8,grouped_shap_rows_fine,10584
9,rq1_rows,324


### Plot manifest by thesis role

,thesis_role,analysis_theme,plot_type,size
0,primary,inertia,pfi_driver_classes,1
1,primary,inertia,pfi_mode_transition_fw0,1
2,primary,inertia,shap_driver_classes,1
3,primary,inertia,shap_mode_transition_fw0,1
4,primary,inertia_fine,pfi_driver_classes_fine,1
5,primary,inertia_fine,pfi_mode_transition_fine_fw0,1
6,primary,inertia_fine,shap_driver_classes_fine,1
7,primary,inertia_fine,shap_mode_transition_fine_fw0,1
8,primary,scope,coverage_heatmap,1
9,primary,static_context,pfi_static_context_pooled,1


,plot_type,path,model_family,regime,building,mode,weather_mode,horizon_h,case_type,feature_name,thesis_role,analysis_theme
0,pfi_driver_classes,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,xgboost,ALL,ALL,ALL_CURRENT,ALL_CURRENT,<NA>,<NA>,<NA>,primary,inertia
1,pfi_mode_transition_fw0,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,xgboost,per_building,ALL,ALL_CURRENT,FW0,<NA>,<NA>,<NA>,primary,inertia
2,shap_driver_classes,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,xgboost,ALL,ALL,ALL_CURRENT,ALL_CURRENT,<NA>,<NA>,<NA>,primary,inertia
3,shap_mode_transition_fw0,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,xgboost,per_building,ALL,ALL_CURRENT,FW0,<NA>,<NA>,<NA>,primary,inertia
4,pfi_driver_classes_fine,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,xgboost,per_building,ALL,ALL_CURRENT,ALL_CURRENT,<NA>,<NA>,<NA>,primary,inertia_fine
5,pfi_mode_transition_fine_fw0,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,xgboost,per_building,ALL,ALL_CURRENT,FW0,<NA>,<NA>,<NA>,primary,inertia_fine
6,shap_driver_classes_fine,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,xgboost,per_building,ALL,ALL_CURRENT,ALL_CURRENT,<NA>,<NA>,<NA>,primary,inertia_fine
7,shap_mode_transition_fine_fw0,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,xgboost,per_building,ALL,ALL_CURRENT,FW0,<NA>,<NA>,<NA>,primary,inertia_fine
8,coverage_heatmap,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,ALL_CURRENT,ALL,ALL,ALL_CURRENT,ALL_CURRENT,<NA>,<NA>,<NA>,primary,scope
9,pfi_static_context_pooled,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,xgboost,pooled_same_buildings,ALL,M4,FW0,<NA>,<NA>,<NA>,primary,static_context
